# F26-07 - Phase 4: Evaluation and Analysis
**AI Sales & Customer Support Agent** (Roman Urdu / English, small fashion shop)

In Phase 3 we built the agent. Here we try to find out how good it really is, and where it breaks.
The notebook has five parts:

* **Part 0** - the agent itself (the Phase 3 code plus the fixes we made in Phase 4). `set_version(1/2/3)` switches between the three versions so we can compare them.
* **Part A** - the test sets: Phase 2 dev/test, our Phase 3 stress set, and 38 new hand-written conversations.
* **Part B** - the RAG checks: retrieval quality, embedding size, and hallucination (answering things the policy does not say).
* **Part C** - metrics and the comparison of approaches.
* **Part D** - error analysis, fairness and robustness.

Files needed in the same folder: catalog.csv, orders.csv, policy.md, policy_rules.json, conversations.json.
Everything is seeded (42), so a full run gives the same numbers.

## Part 0 - The agent (Phase 3 code + Phase 4 fixes)
Versions: **v1** = what we handed in for Phase 3. **v2** = fixes for the mistakes the Phase 3 test and stress sets showed.
**v3** = fixes for the mistakes the new hand-written set showed. Because of that, a version's score is only blind on a set it
was not fixed against. We say which one is which in every table.

### 1. Imports, settings, data

In [1]:
import json, re, os, datetime, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.pipeline import make_pipeline, make_union
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
V3 = True
V2 = True      # Phase 4 fixes on. set_version(1) brings back the exact Phase 3 behaviour (used for the before/after comparison)
DATA_DIR = os.environ.get("DATA_DIR", ".")
TODAY = datetime.date(2026, 10, 17)          # same "today" as the Phase 2 order data


def P(name):
    return os.path.join(DATA_DIR, name)


catalog = pd.read_csv(P("catalog.csv"), dtype={"size": str})
catalog["color"] = catalog["color"].str.strip().str.title()
catalog["size"] = catalog["size"].str.strip().str.upper().replace({"FREE": "Free"})
orders = pd.read_csv(P("orders.csv"), dtype=str)
orders["phone"] = orders["phone"].str.zfill(11)
orders["qty"] = orders["qty"].astype(int)
orders["delivered_date"] = pd.to_datetime(orders["delivered_date"])
policy_text = open(P("policy.md"), encoding="utf-8").read()
rules = json.load(open(P("policy_rules.json"), encoding="utf-8"))
conversations = json.load(open(P("conversations.json"), encoding="utf-8"))["conversations"]
dev = [c for c in conversations if c["split"] == "dev"]
test = [c for c in conversations if c["split"] == "test"]
SKU = catalog.set_index("sku")
print(f"catalog={len(catalog)}  orders={len(orders)}  conversations={len(conversations)} (dev {len(dev)}, test {len(test)})")

catalog=572  orders=51  conversations=172 (dev 120, test 52)


### 2. Text cleaning and lexicons (colors, sizes, products, cities)

In [2]:
URDU_WORDS = set("""hai hain mein main me ka ki ke ko se par pe aur ya kya kitna kitne kitni kab kahan chahiye chahta chahti karo kar karna
karni do dena de bhej bhejo batao bata dikhao nahi haan ji acha theek sahi bhai mera meri mere mujhe aap apka wo ye ab abhi
bohat bahut mil milega hoga ho tak din paise wapis kal aaj naam jootay kapre insan baat kharab phat gaya diya""".split())

NUMWORDS = {"ek": 1, "do": 2, "teen": 3, "char": 4, "paanch": 5, "panch": 5, "chay": 6, "saat": 7, "aath": 8, "nau": 9, "das": 10}
SIZE_ALIAS = {"s": "S", "small": "S", "smll": "S", "m": "M", "medium": "M", "mdm": "M", "meedium": "M",
              "l": "L", "large": "L", "lrg": "L", "xl": "XL"}
COLOR_ALIASES = {"kala": "Black", "kali": "Black", "blk": "Black", "blak": "Black", "safaid": "White", "safed": "White",
                 "wht": "White", "navy blue": "Navy", "navi": "Navy", "gray": "Grey", "slate": "Grey", "marun": "Maroon",
                 "gulabi": "Pink", "sabz": "Green", "hara": "Green", "asmani": "Sky Blue", "laal": "Red", "nila": "Blue"}
for _c in catalog["color"].unique():
    COLOR_ALIASES[_c.lower()] = _c
COLOR_FUZZ_KEYS = [k for k in COLOR_ALIASES if len(k) >= 5 and " " not in k]

CITY_LIST = sorted(set(rules["delivery"]["tier_a_cities"] + list(orders["city"].unique()) + [
    "Karachi", "Peshawar", "Multan", "Quetta", "Hyderabad", "Sargodha", "Bahawalpur", "Sukkur", "Larkana", "Gujrat",
    "Sahiwal", "Okara", "Mardan", "Abbottabad", "Jhelum", "Kasur", "Rahim Yar Khan", "Dera Ghazi Khan", "Mirpur"]),
    key=len, reverse=True)
TIER_A = {c.lower() for c in rules["delivery"]["tier_a_cities"]}

PHONE_RE = re.compile(r"(?<!\d)(?:\+?92|0)?[-\s]?(3\d{2})[-\s]?(\d{7})(?!\d)")
ADDR_RE = re.compile(r"\b(house|street|flat|plot|shop|gali|phase|sector|block)\s*(?:no\.?|#)?\s*\d")


def norm(text):
    """Lowercase + normalise spelling variants of product words (t-shirt, 3-piece, co-ord, 2-3Y ...)."""
    t = text.lower().replace("&", " and ")
    t = re.sub(r"[\u200b\U0001F300-\U0001FAFF]", "", t)
    t = re.sub(r"\bt[\s-]?shirt\b", "tshirt", t)
    t = re.sub(r"\b([23])[\s-]?piece\b" if V2 else r"\b(\d)[\s-]?piece\b", r"\1piece", t)
    t = re.sub(r"\bco[\s-]?ord\b", "coord", t)
    t = re.sub(r"\bextra[\s-]?large\b", "xl", t)
    t = re.sub(r"\b(\d)\s*-\s*(\d)\s*y\b", r"\1-\2y", t)
    t = re.sub(r"\bwash\s+n\s+wear\b", "wash and wear", t)
    return re.sub(r"\s+", " ", t).strip()


def tokens(t):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", t)


def strip_phones(text):
    return PHONE_RE.sub(" ", text)


def find_phone(text):
    m = PHONE_RE.search(text)
    return "0" + m.group(1) + m.group(2) if m else None


def clean_text(text):
    """Phase 2 cleaning: lowercase, joined phone numbers, no emojis / extra spaces."""
    t = re.sub(r"\b(03\d{2})[-\s](\d{7})\b", r"\1\2", text.lower().strip())
    t = re.sub(r"[\u200b\U0001F300-\U0001FAFF]", "", t)
    return re.sub(r"\s+", " ", t).strip()


# --- products: token coverage with fuzzy token matching -----------------------------------
PRODUCTS = sorted(catalog["name"].unique())
PROD_TOK = {p: [w for w in tokens(norm(p)) if w != "and"] for p in PRODUCTS}


def tok_match(word, toks):
    for m in toks:
        if word == m:
            return True
        if len(word) >= 5 and len(m) >= 5 and abs(len(word) - len(m)) <= 1 and fuzz.ratio(word, m) >= 85:
            return True
    return False


def find_product(toks, color=None, size=None):
    scored = []
    for p, pt in PROD_TOK.items():
        hit = sum(tok_match(w, toks) for w in pt)
        cov = hit / len(pt)
        if hit >= 2 and cov >= 0.6:
            scored.append((cov, hit, p))
    if not scored:
        return None
    best = max((s[0], s[1]) for s in scored)
    cands = [s[2] for s in scored if (s[0], s[1]) == best]
    if len(cands) > 1:                                    # tie: prefer a product that really has that color/size
        ok = [p for p in cands if not catalog[(catalog.name == p) &
                                              ((catalog.color == color) if color else True) &
                                              ((catalog["size"] == size) if size else True)].empty]
        cands = ok or cands
    return sorted(cands)[0]


def find_colors(t):
    toks, found, i = tokens(t), [], 0
    while i < len(toks):
        two = toks[i] + " " + toks[i + 1] if i + 1 < len(toks) else None
        if two in COLOR_ALIASES:
            found.append(COLOR_ALIASES[two]); i += 2; continue
        w = toks[i]
        if w in COLOR_ALIASES:
            found.append(COLOR_ALIASES[w])
        elif len(w) >= 5:
            m = process.extractOne(w, COLOR_FUZZ_KEYS, scorer=fuzz.ratio, score_cutoff=86)
            if m:
                found.append(COLOR_ALIASES[m[0]])
        i += 1
    return found


def find_size(t):
    m = re.search(r"\b(\d)-(\d)y\b", t)
    if m:
        return f"{m.group(1)}-{m.group(2)}Y"
    m = re.search(r"\bsize\s*(?:number|no)?\s*(\d{1,2}|xl|s|m|l|small|medium|large|mdm|smll|lrg)\b", t)
    if m:
        g = m.group(1)
        return SIZE_ALIAS.get(g, g)
    m = re.search(r"\b(\d{2})\s*size\b", t)
    if m:
        return m.group(1)
    for w in tokens(t):
        if w in SIZE_ALIAS:
            return SIZE_ALIAS[w]
    return None


def find_qty(t):
    t = re.sub(r"size\s*(?:number|no)?\s*\d{1,2}|\b\d{1,2}\s*size|\b\d-\d y\b", " ", t)
    toks = tokens(t)
    for i, w in enumerate(toks):
        prev = toks[i - 1] if i else ""
        nxt = toks[i + 1] if i + 1 < len(toks) else ""
        if w.isdigit():
            if len(w) <= 2 and nxt not in ("din", "days", "baje", "minute", "hafte", "mahine", "dinon"):
                return int(w)
        elif w in NUMWORDS:
            if w == "do" and not (i == 0 or prev in ("aur", "ya", "phir")):
                continue                                   # "kar do", "bhej do" are not the number two
            return NUMWORDS[w]
    return None


def find_city(raw):
    low, best = raw.lower(), None
    for c in CITY_LIST:
        for m in re.finditer(r"\b" + re.escape(c.lower()) + r"\b", low):
            if re.match(r"\s+road\b", low[m.end():]):      # "Lahore Road, Sheikhupura" is a street name
                continue
            if best is None or m.start() >= best[0]:
                best = (m.start(), c)
    return best[1].title() if best else None


def find_name(raw):
    m = re.search(r"(?:naam|name)\s*[:\-]?\s*(?:hai\s+)?([A-Za-z]+(?:\s+[A-Za-z]+){0,2}?)(?=\s+hai\b|\s*[,\n]|\s+(?:number|phone|no)\b|\s*\d|$)",
                  raw, flags=re.I)
    if m:
        return m.group(1).strip().title()
    m = re.search(r"[-\u2013]\s*([A-Z][a-z]+(?:\s[A-Z][a-z]+)+)\s*[-\u2013]", raw)
    if m:
        return m.group(1)
    m = re.match(r"\s*([A-Z][a-z]+(?:\s[A-Z][a-z]+)+)\s*,?\s*\+?\d", raw)
    return m.group(1) if m else None


CATEGORY_KEYS = [("kurti", "Women Kurti"), ("kurta", "Men Kurta"), ("tshirt", "Men Western"), ("shirt", "Men Western"),
                 ("jeans", "Men Western"), ("hoodie", "Men Winter"), ("sweater", "Men Winter"), ("jootay", "Footwear"),
                 ("jutay", "Footwear"), ("shoes", "Footwear"), ("abaya", "Women Modest"), ("hijab", "Women Modest")]


def find_category(toks):
    for key, cat in CATEGORY_KEYS:
        if key in toks:
            if cat == "Men Winter" and ("women" in toks or "ladies" in toks):
                return "Women Winter"
            return cat
    return None


def extract(raw):
    """Turn one raw customer message into entities."""
    t = norm(raw)
    tn = strip_phones(t)
    toks = tokens(tn)
    phone = find_phone(t)
    details = bool(phone) or bool(ADDR_RE.search(t)) or bool(re.search(r"\b(naam|name|address)\s*[:\-]", t))
    ids = re.findall(r"(?<!\d)(\d{5})(?!\d)", tn)
    e = {"phone": phone, "order_id": ids[0] if ids else None, "city": find_city(raw), "is_details": details,
         "name": find_name(raw) if details else None,
         "address": raw.strip() if (ADDR_RE.search(t) or re.search(r"address\s*[:\-]", t)) else None,
         "product": None, "color": None, "size": None, "qty": None, "extra": None, "budget": None, "category": None}
    if re.search(r"\b(naam|name|address)\s*[:\-]", t) or ADDR_RE.search(t):
        return e                                           # name/address messages carry no item information
    colors = find_colors(tn)
    e["size"] = find_size(tn)
    e["color"] = colors[0] if colors else None
    e["product"] = find_product(toks, e["color"], e["size"])
    qty_text = tn
    if V3 and e["product"] and not e["size"]:                # "chino trouser khaki 32": a bare number that is a real size of that product
        sizes = set(catalog.loc[catalog["name"] == e["product"], "size"])
        nums = [w for w in toks if w.isdigit() and w in sizes]
        if nums:
            e["size"] = nums[0]
            qty_text = re.sub(r"(?<!\d)" + nums[0] + r"(?!\d)", " ", tn, count=1)
    e["qty"] = find_qty(qty_text)
    e["category"] = find_category(toks)
    for w in toks:                                         # budget: a 3-4 digit number
        if w.isdigit() and 300 <= int(w) <= 99999 and len(w) <= 4:
            e["budget"] = int(w)
    m = re.search(r"(\d+|ek|do|teen|char)\s+same\b.*?\baur\s+(?:(\d+|ek|do|teen)\s+)?([a-z]+(?:\s+blue)?)", tn)
    if m:                                                  # "2 same bhej do aur ek white"
        c2 = find_colors(m.group(3))
        if c2:
            first = int(m.group(1)) if m.group(1).isdigit() else NUMWORDS[m.group(1)]
            q2 = m.group(2)
            e["qty"] = first
            e["extra"] = {"color": c2[0], "qty": (int(q2) if q2 and q2.isdigit() else NUMWORDS.get(q2, 1))}
    return e


demo = ["Salam, black linen kurta M mein hai?", "2 same bhej do aur ek white", "teen navy khussa embroidered size 37 chahiye",
        "Mera naam Ali Raza hai, 0300-1234567, house 12 street 4 Model Town Lahore"]
for d in demo:
    print(d, "->", {k: v for k, v in extract(d).items() if v})

Salam, black linen kurta M mein hai? -> {'product': 'Linen Kurta', 'color': 'Black', 'size': 'M', 'category': 'Men Kurta'}
2 same bhej do aur ek white -> {'color': 'White', 'qty': 2, 'extra': {'color': 'White', 'qty': 1}}
teen navy khussa embroidered size 37 chahiye -> {'product': 'Khussa Embroidered', 'color': 'Navy', 'size': '37', 'qty': 3}
Mera naam Ali Raza hai, 0300-1234567, house 12 street 4 Model Town Lahore -> {'phone': '03001234567', 'city': 'Lahore', 'is_details': True, 'name': 'Ali Raza', 'address': 'Mera naam Ali Raza hai, 0300-1234567, house 12 street 4 Model Town Lahore'}


### 3. Policy RAG: chunking, embeddings (TF-IDF + SVD), retrieval

In [3]:
def chunk_policy(text, max_words=250):
    chunks = []
    for section in re.split(r"\n(?=## )", text):
        lines = [l for l in section.strip().split("\n") if l.strip()]
        if not lines or not lines[0].startswith("## "):
            continue
        title, body = lines[0][3:].strip(), lines[1:]
        sec_no = int(re.match(r"(\d+)", title).group(1))
        cur, cnt = [], 0
        for line in body:
            w = len(line.split())
            if cur and cnt + w > max_words:
                chunks.append({"section": sec_no, "title": title, "text": " ".join(cur)}); cur, cnt = [], 0
            cur.append(line.strip()); cnt += w
        if cur:
            chunks.append({"section": sec_no, "title": title, "text": " ".join(cur)})
    return chunks


policy_chunks = [c for c in chunk_policy(policy_text) if c["section"] <= 9]      # section 10 (FAQ) is used to expand sections
FAQ_TO_SECTION = {"delivery kitne din": 2, "cod available": 3, "delivery charges kitne": 2, "exchange ho sakta": 6,
                  "sale wale item": 6, "refund milega": 7, "mera order kahan": 5}
faq_questions = re.findall(r"\*\*(.+?)\*\*", policy_text)


class PolicyRAG:
    """Embeds policy chunks with char n-gram TF-IDF (robust to Roman Urdu spelling) and a dense SVD (LSA) layer."""

    def __init__(self, chunks, use_faq=True, alpha=0.5):
        self.chunks, self.alpha = chunks, alpha
        docs = []
        for c in chunks:
            extra = " ".join(q for q in faq_questions
                             if use_faq and any(k in q.lower() and s == c["section"] for k, s in FAQ_TO_SECTION.items()))
            docs.append(norm(c["title"] + " " + c["text"] + " " + extra))
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
        self.X = self.vec.fit_transform(docs)
        self.svd = TruncatedSVD(n_components=min(8, self.X.shape[0] - 1), random_state=SEED)
        self.E = normalize(self.svd.fit_transform(self.X))

    def retrieve(self, query, k=3):
        q = self.vec.transform([norm(query)])
        sparse = (self.X @ q.T).toarray().ravel()
        dense = self.E @ normalize(self.svd.transform(q)).ravel()
        score = self.alpha * sparse + (1 - self.alpha) * dense
        idx = np.argsort(-score)[:k]
        return [(self.chunks[i], float(score[i]), float(sparse[i])) for i in idx]


rag = PolicyRAG(policy_chunks)
print(f"{len(policy_chunks)} policy chunks, embedding matrix {rag.E.shape}")
for q in ["Karachi delivery kitne din mein hoti hai?", "sale ka suit exchange hoga?", "refund milega kya?"]:
    top = rag.retrieve(q, 1)[0]
    print(f"{q!r} -> section {top[0]['title']} (score {top[1]:.2f})")

9 policy chunks, embedding matrix (9, 8)
'Karachi delivery kitne din mein hoti hai?' -> section 2. Delivery Areas and Charges (score 0.64)
'sale ka suit exchange hoga?' -> section 6. Exchange Policy (score 0.64)
'refund milega kya?' -> section 7. Returns and Refunds (score 0.63)


### 4. Guardrail layer (reads policy_rules.json) and tools

In [4]:
def delivery_charge(city, product_total):
    d = rules["delivery"]
    if product_total >= d["free_delivery_min_product_total"]:
        return 0
    return d["tier_a_charge"] if city and city.lower() in TIER_A else d["default_charge"]


def eta_days(city):
    d = rules["delivery"]
    return d["eta_days_tier_a"] if city and city.lower() in TIER_A else d["eta_days_other"]


class Store:
    """Mutable copy of catalog stock + orders, one per conversation. Every tool call is written to an audit log."""

    def __init__(self):
        self.stock = dict(zip(catalog["sku"], catalog["stock"]))
        self.orders = orders.copy()
        self.next_id = 20001
        self.audit = []

    def _log(self, tool, args, out):
        self.audit.append({"tool": tool, "args": args, "out": out})
        return out

    def variants(self, product, color=None, size=None):
        df = catalog[catalog["name"] == product]
        if color:
            df = df[df["color"] == color]
        if size:
            df = df[df["size"] == size]
        return df

    def alternatives(self, product, color, size, n=4):
        df = self.variants(product)
        df = df[[self.stock[s] > 0 for s in df["sku"]]]
        same = df[(df["color"] == color) | (df["size"] == size)]
        return [(r.color, r.size, int(self.stock[r.sku])) for r in same.head(n).itertuples()]

    def check_stock(self, product, color, size, qty=None):
        rows = self.variants(product, color, size)
        if rows.empty:
            out = {"found": False, "in_stock": False, "available": 0, "suggest_alternatives": True,
                   "alternatives": self.alternatives(product, color, size)}
        else:
            r = rows.iloc[0]
            avail = int(self.stock[r["sku"]])
            out = {"found": True, "sku": r["sku"], "in_stock": avail > 0, "available": avail, "price": int(r["price"]),
                   "is_sale": bool(r["is_sale"])}
            if avail == 0:
                out["suggest_alternatives"] = True
                out["alternatives"] = self.alternatives(product, color, size)
            elif qty and qty > avail:
                out["requested_exceeds_stock"] = True
        return self._log("check_stock", (product, color, size, qty), out)

    def reserve_order(self, lines):
        """lines = [(sku, qty)]. Reserves stock (all or nothing)."""
        if any(self.stock[s] < q for s, q in lines):
            return self._log("reserve_order", lines, False)
        for s, q in lines:
            self.stock[s] -= q
        return self._log("reserve_order", lines, True)

    def create_order(self, lines, name, phone, city, address, product_total, charge):
        oid = str(self.next_id); self.next_id += 1
        for s, q in lines:
            self.orders.loc[len(self.orders)] = [oid, name, phone, s, q, city, "Placed", str(TODAY), pd.NaT, None, None,
                                                 str(product_total + charge)]
        out = {"order_id": oid, "product_total": product_total, "delivery_charge": charge,
               "grand_total": product_total + charge, "payment": "COD"}
        return self._log("create_order", (oid, name, phone, city), out)

    def get_order(self, order_id):
        df = self.orders[self.orders["order_id"] == str(order_id)]
        return None if df.empty else df.iloc[0]

    def get_order_status(self, order_id, phone):
        o = self.get_order(order_id)
        if o is None:
            return self._log("get_order_status", (order_id, phone), {"found": False})
        if o["phone"] != phone:
            return self._log("get_order_status", (order_id, "***"), {"found": True, "share_details": False})
        out = {"found": True, "share_details": True, "status": o["status"]}
        if o["status"] in ("Shipped", "Delivered"):
            out.update(courier=o["courier"], tracking_no=o["tracking_no"])
        if o["status"] == "Delivered":
            out["delivered_date"] = str(o["delivered_date"].date())
        return self._log("get_order_status", (order_id, "***"), out)


def exchange_check(store, order, target_size, target_color, different_product):
    """Policy section 6. Returns (ok, reason, details). The first failed condition decides the escalation reason."""
    ex = rules["exchange"]
    item = SKU.loc[order["sku"]]
    if different_product and ex["same_product_only"]:
        return False, "different_product_requested", {}
    if item["category"] in ex["non_exchangeable_categories"]:
        return False, "category_not_exchangeable", {}
    if item["is_sale"] and not ex["sale_items_allowed"]:
        return False, "sale_item_not_exchangeable", {}
    if order["status"] != "Delivered":
        return False, "order_not_delivered", {}
    days = (pd.Timestamp(TODAY) - order["delivered_date"]).days
    if days > ex["window_days"]:
        return False, "outside_7_day_window", {"days_since_delivery": days}
    color, size = target_color or item["color"], target_size or item["size"]
    rows = store.variants(item["name"], color, size)
    if rows.empty or store.stock[rows.iloc[0]["sku"]] <= 0:
        return False, "replacement_out_of_stock", {"days_since_delivery": days}
    return True, None, {"eligible": True, "days_since_delivery": days, "replacement_sku": rows.iloc[0]["sku"],
                        "exchange_courier_charge": ex["exchange_courier_charge"], "product": item["name"],
                        "color": color, "size": size}

### 5. Intent routing: keyword/regex rules, TF-IDF + Logistic Regression, and the hybrid

In [5]:
HANDOFF_RULES = [
    ("angry_customer_legal_threat", r"court|wakeel|lawyer|legal|police|\bfia\b"),
    ("angry_customer", r"bakwas|fraud|bekar|ghatiya|bewakoof|cheat|dhoka|loot|worst|thug|pagal"),
    ("customer_requested_human", r"insan se|owner se|\bhuman\b|real banda|banda chahiye|kisi se baat|manager se|agent se"),
    ("wrong_item_received", r"wrong item|galat (item|cheez|size|color)|mangwaya tha.*bhej|bhej diya"),
    ("damaged_item_refund", r"(phat|kharab|damaged|toota|defective|khuli hui|torn).*(refund|paise wapis|wapis karo|return)|(refund|paise wapis|wapis karo|return).*(phat|kharab|damaged|toota|defective|khuli hui|torn)"),
    ("refund_request", r"refund|paise wapis|wapis karo|\breturn\b"),
    ("damaged_item_refund", r"\bphat\b|phat gaya|kharab|damaged|toota|defective|khuli hui|\btorn\b"),
    ("order_cancellation_request", r"cancel"),
    ("order_change_request", r"(address|number|phone|order|quantity|qty) (change|badal)|change (address|order)|address badal"),
    ("discount_request", r"discount|last price|kam kar|price kam|bargain|sasta|best price|bulk deal|rate kam|kuch kam"),
    ("online_payment_request", r"jazzcash|easypaisa|bank transfer|advance payment|account number|online payment"),
]
ORDER_VERBS = r"\border\b|bhej|kar do|karna hai|mangwa|\bbook\b|confirm|le lunga|lena hai|purchase|\bbuy\b"
POLICY_WORDS = (r"delivery|parcel|pohanch|charge|\bcod\b|cash on delivery|kitne din|staff|reply|showroom|shop kahan|store|"
                r"bahar ke mulk|abroad|outside pakistan|international|timing|kab tak|paise lagen")


V2_HANDOFF_EXTRA = [
    ("customer_requested_human", r"insan ko|bot se nahi|real person|kisi insan"),
    ("angry_customer", r"bura experience|bohat bura|disappointed|shikayat|pareshan"),
]
OUTSIDE_V2 = (r"polyester|shrink|dhulai|gift wrap|message card|custom stitching|alteration|tailoring|warranty|"
              r"fabric (quality|kaisa|kaisi|pakka)|material (kaisa|kaisi|kya)")


V3_HANDOFF_EXTRA = [("refund_request", r"order wap[ai]s|wap[ai]s (karna|kar do|karo|chahiye|bhej)|paise wap[ai]s")]


def handoff_reason(t):
    for reason, rx in (V2_HANDOFF_EXTRA if V2 else []) + HANDOFF_RULES + (V3_HANDOFF_EXTRA if V3 else []):
        if re.search(rx, t):
            if reason == "order_change_request" and "exchange" in t:
                continue
            return reason
    for kw in rules["escalation_keywords"]:                   # safety net: any keyword from policy_rules.json
        if re.search(r"\b" + re.escape(kw) + r"\b", t):
            return "needs_human"
    return None


def rule_router(text):
    """Returns (intent, reason). intent in handoff/exchange/policy/order_status/order/stock/recommend or None."""
    t = norm(strip_phones(text))
    toks = tokens(t)
    h = handoff_reason(t)
    if h:
        return "handoff", h
    if V2 and re.search(OUTSIDE_V2, t):
        return "outside", None
    has_oid = bool(re.search(r"(?<!\d)\d{5}(?!\d)", t))
    colors, size = find_colors(t), find_size(t)
    prod = find_product(toks, colors[0] if colors else None, size)
    exch_rx = r"\bbadal\w*|\bbadl\w*|\bbadle\b|\bswap\b" if V3 else r"\bbadal\b|\bbadle\b|\bswap\b"
    if "exchange" in t or re.search(exch_rx, t):
        pol_rx = r"policy|hoga|hogi|ho sakta|ho sakti|kitne din" + (r"|ho jata|ho jati|hota hai|hoti hai" if V3 else "")
        if re.search(pol_rx, t) and not has_oid:
            return "policy", None
        if V3 and re.search(r"\bkya\b|\?", norm(text)) and not has_oid and not re.search(r"karna|kar do|chahiye|badal", t):
            return "policy", None
        return "exchange", None
    if (has_oid or "order" in toks) and re.search(r"status|track|kahan|kidhar|pohanch|where", t) and not prod:
        return "order_status", None
    if has_oid and not prod and "order" in toks and not re.search(ORDER_VERBS.replace(r"\border\b|", ""), t):
        return "order_status", None
    has_details = bool(PHONE_RE.search(norm(text))) or bool(ADDR_RE.search(t))
    if prod:
        return ("order" if (re.search(ORDER_VERBS, t) or has_details) else "stock"), None
    budget = [w for w in toks if w.isdigit() and 300 <= int(w) <= 9999 and len(w) <= 4]
    if budget:
        return "recommend", None
    if re.search(POLICY_WORDS, t):
        return "policy", None
    return None, None


COARSE = {"place_order": "order", "check_stock": "stock", "exchange": "exchange", "policy_question": "policy",
          "order_status": "order_status", "recommend": "recommend", "refund": "handoff", "complaint": "handoff",
          "human_request": "handoff", "payment_request": "handoff", "change_order": "handoff",
          "discount_request": "handoff", "cancel_order": "handoff", "other": "outside", "unknown": "unknown"}


def mask(text):
    t = norm(strip_phones(text))
    return re.sub(r"(?<!\d)\d{5}(?!\d)", " oid ", t)


def conv_text(conv, upto=None):
    turns = conv["turns"] if upto is None else conv["turns"][:upto]
    return " ".join(t["text"] for t in turns if t["speaker"] == "customer")


def make_ml():
    return make_pipeline(
        make_union(TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True),
                   TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)),
        LogisticRegression(C=20, max_iter=3000, class_weight="balanced", random_state=SEED))


# the ML intent model is trained on DEV conversations only
ml_intent = make_ml().fit([mask(conv_text(c)) for c in dev], [COARSE[c["expected"]["intent"]] for c in dev])


def ml_router(text, min_prob=0.0):
    proba = ml_intent.predict_proba([mask(text)])[0]
    i = int(np.argmax(proba))
    return (ml_intent.classes_[i], float(proba[i])) if proba[i] >= min_prob else (None, float(proba[i]))


def question_like(text):
    t = norm(strip_phones(text))
    return len(tokens(t)) >= 5 and ("?" in text or re.search(r"\bkya\b|\bkaise\b|\bkyun\b", t) is not None)


RAG_MIN = 0.10      # minimum retrieval score for answering from the policy (chosen on dev)


def hybrid_router(text, rag_obj=None):
    intent, reason = rule_router(text)
    if intent:
        return intent, reason
    cls, p = ml_router(text, 0.55)
    if cls in ("order", "stock", "exchange", "order_status", "recommend", "policy", "handoff"):
        return cls, ("needs_human" if cls == "handoff" else None)
    if question_like(text):
        sc = (rag_obj or rag).retrieve(text, 1)[0][2]
        return ("policy", None) if sc >= RAG_MIN else ("outside", None)
    return "unknown", None


def rules_only_router(text):
    intent, reason = rule_router(text)
    if intent:
        return intent, reason
    return ("outside", None) if question_like(text) else ("unknown", None)


def ml_only_router(text):
    cls, _ = ml_router(text)
    return cls, ("needs_human" if cls == "handoff" else None)


def set_version(v):
    """v1 = Phase 3 system. v2 = fixes from the Phase 3 test/stress errors. v3 = v2 + fixes from the new hand-written set
    (bare numeric sizes, policy phrasing, 'badalna', 'wapas', and the retrieval-score threshold calibrated on dev)."""
    global V2, V3, RAG_MIN, ml_intent
    V2, V3 = v >= 2, v >= 3
    RAG_MIN = 0.115 if V3 else 0.10
    ml_intent = make_ml().fit([mask(conv_text(c)) for c in dev], [COARSE[c["expected"]["intent"]] for c in dev])


set_version(3)
print("ML intent classes:", list(ml_intent.classes_))

ML intent classes: [np.str_('exchange'), np.str_('handoff'), np.str_('order'), np.str_('order_status'), np.str_('outside'), np.str_('policy'), np.str_('recommend'), np.str_('stock'), np.str_('unknown')]


### 6. Dialogue manager: session state, system prompt, agent (tool calling + guardrails + handoff)

In [6]:
SYSTEM_PROMPT = """You are the customer-support and sales assistant of Saira Fashion House, an online-only fashion store in Pakistan.
Customers write in Roman Urdu, English or a mix. Reply in the customer's language mix, short and polite.
RULES (from policy.md, enforced again in code by the guardrail layer):
1. Use ONLY facts returned by the tools and the retrieved policy text. Never invent prices, stock, delivery times or policies.
2. Before creating an order you need product, color, size, quantity, name, phone, address and city. Show the order summary
   (items, product total, delivery charges, grand total, COD) and wait for a clear confirmation.
3. Share order status only when both the order number and the phone number match.
4. Never approve refunds, returns for money or compensation. Never promise delivery dates (only the ranges in the policy).
5. Hand over to a human (do not guess) for: refund/return/damage, cancel or change of a confirmed order, discount or bargaining,
   angry customers or legal threats, a request for a human, quantity above 10, COD above Rs. 25,000, online payment,
   exchange that fails any rule, order not found, anything outside the policy, or an unclear request after two clarifications.
6. When handing over, tell the customer that staff will reply daily between 10:00 AM and 8:00 PM.
"""
USE_LLM = False      # optional: let an LLM re-phrase the verified reply (see llm_polish); decisions never depend on it


def llm_polish(facts_reply, history, system_prompt=SYSTEM_PROMPT):
    """Hook for an LLM (OpenAI / Gemini / Claude API). Not used in the reported results: decisions and numbers always come from
    the tools; an LLM may only re-phrase `facts_reply`. Left as a stub because no API key is shipped with the project."""
    raise NotImplementedError("plug in your LLM client here")


CONFIRM_RE = re.compile(r"\b(yes|haan|han|ji|confirm|confirmed|done|sahi|theek|thik|bilkul|ok|okay|kar do|bhej do)\b")
DENY_RE = re.compile(r"\b(nahi|nahin|no|cancel|ruko|wait|mat)\b")


def rs(n):
    return f"Rs. {int(n):,}"


class Session:
    def __init__(self):
        self.items, self.texts = [], []
        self.name = self.phone = self.address = self.city = self.order_id = None
        self.budget = self.cat_hint = None
        self.target_size = self.target_color = None
        self.different_product = False
        self.summary_shown = False
        self.escalated = self.escalated_result = None
        self.clarify = 0
        self.lang = None
        self.last_intent = None
        self.exchange_item = None

    def update(self, raw):
        e = extract(raw)
        self.texts.append(raw)
        toks = set(tokens(norm(raw)))
        if toks & URDU_WORDS:
            self.lang = "ru"
        elif self.lang is None:
            self.lang = "en"
        for k in ("phone", "order_id", "city", "name", "address"):
            if e[k]:
                setattr(self, k, e[k])
        if re.search(r"ki jagah|ke bajaye|instead of|badle mein", norm(raw)):
            self.different_product = True
        if e["is_details"] and not (e["product"] or e["color"] or e["size"] or e["qty"]):
            return e
        if e["budget"]:
            self.budget = e["budget"]
        if e["category"]:
            self.cat_hint = e["category"]
        if e["size"]:
            self.target_size = e["size"]
        if e["color"]:
            self.target_color = e["color"]
        last = self.items[-1] if self.items else None
        if e["product"] and (last is None or last["product"] != e["product"]):
            self.items = [{"product": e["product"], "color": e["color"], "size": e["size"], "qty": e["qty"]}]
            self.summary_shown = False
        elif last is not None:
            changed = False
            for k in ("color", "size", "qty"):
                if e[k] is not None and not (k == "color" and e["extra"]) and last[k] != e[k]:
                    last[k], changed = e[k], True
            if changed:
                self.summary_shown = False
        if e["extra"] and self.items:
            base = dict(self.items[0])
            base["color"], base["qty"] = e["extra"]["color"], e["extra"]["qty"]
            self.items = [self.items[0], base]
            self.summary_shown = False
        return e

    def entities(self):
        rec = self.last_intent == "recommend"
        items = [dict(i) for i in self.items] if self.last_intent in ("stock", "order", "handoff") else []
        if self.last_intent == "exchange" and self.exchange_item:
            items = [self.exchange_item]
        return {"items": items, "location": self.city, "order_id": self.order_id, "phone": self.phone,
                "budget": self.budget if rec else None, "category": self.cat_hint if rec else None}


class Agent:
    def __init__(self, router="hybrid", guardrails=True):
        self.router, self.guardrails = router, guardrails
        self.s, self.store, self.trace = Session(), Store(), []

    # ---------- helpers
    def say(self, ru, en):
        return ru if self.s.lang == "ru" else en

    def act(self, action, reply, result=None, escalate=False, reason=None):
        out = {"action": action, "reply": reply, "result": result or {}, "escalate": escalate, "reason": reason}
        self.trace.append(out)
        return out

    def escalate(self, reason, result=None):
        s = self.s
        s.escalated, s.escalated_result = reason, result or {}
        owner_summary = {"customer": s.name or s.phone, "order_id": s.order_id, "issue": " | ".join(s.texts), "reason": reason}
        self.store._log("escalate", reason, owner_summary)
        return self.act("escalate", self.say(
            "Aap ki request hamare staff ko bhej di gayi hai. Staff roz subah 10 se raat 8 baje ke darmiyan aap ko reply karega.",
            "I have passed your request to our staff. They will reply daily between 10 AM and 8 PM."),
            result, True, reason)

    # ---------- routing
    def route(self, cum):
        fn = {"hybrid": hybrid_router, "rules": rules_only_router, "ml": ml_only_router}[self.router]
        intent, reason = fn(cum)
        if self.guardrails and intent != "handoff" and any(
                i["qty"] and i["qty"] > rules["order"]["max_qty_per_item"] for i in self.s.items):
            return "handoff", "quantity_above_limit"
        return intent, reason

    def handle(self, raw):
        s = self.s
        s.update(raw)
        if s.escalated:                                              # once handed over, the AI stays out of the decision
            return self.act("escalate", self.say("Aap ki request staff ke paas hai, wo jald reply karenge.",
                                                 "Your request is with our staff; they will reply soon."),
                            s.escalated_result, True, s.escalated)
        intent, reason = self.route(" ".join(s.texts))
        s.last_intent = intent
        if intent == "handoff":
            return self.escalate(reason or "needs_human")
        if intent == "outside":
            return self.escalate("outside_policy")
        if intent in ("stock", "order") and not s.items:
            intent = "unknown"
        if intent == "unknown":
            s.clarify += 1
            if s.clarify > 2:
                return self.escalate("unclear_after_two_attempts")
            return self.act("clarify", self.say("Maazrat, mujhe samajh nahi aaya. Kya aap product ka naam, color aur size bata sakte hain?",
                                                "Sorry, I did not understand. Could you tell me the product, color and size?"))
        return {"policy": self.do_policy, "recommend": self.do_recommend, "stock": self.do_stock, "order": self.do_order,
                "order_status": self.do_status, "exchange": self.do_exchange}[intent](raw)

    # ---------- policy questions (RAG + rule facts)
    def do_policy(self, raw):
        s = self.s
        t = norm(" ".join(s.texts))
        d, topics, facts = rules["delivery"], set(), {}
        if "exchange" in t:
            topics.add("exchange")
        elif re.search(r"kitne din|pohanch|\beta\b|how many days", t):
            topics.add("eta")
        if re.search(r"charge|paise|kitna lagega", t):
            topics.add("charge")
        if re.search(r"\bcod\b|cash on delivery", t):
            topics.add("cod")
        if re.search(r"staff|reply|timing|hours", t):
            topics.add("hours")
        if re.search(r"showroom|shop kahan|store|walk", t):
            topics.add("online")
        if re.search(r"bahar ke mulk|abroad|outside pakistan|international|overseas", t):
            topics.add("abroad")
        if re.search(r"free delivery|delivery free", t):
            topics.add("free")
        if not topics and "delivery" in t:
            topics |= {"eta", "charge"}
        top = rag.retrieve(" ".join(s.texts), 2)
        source = top[0][0]["title"]
        if not topics:
            if top[0][2] < RAG_MIN:
                return self.escalate("outside_policy")
            return self.act("answer_policy", self.say("Policy ke mutabiq: ", "According to our policy: ") + top[0][0]["text"][:350],
                            {"source": source})
        parts = []
        if "eta" in topics:
            facts["eta_days"] = eta_days(s.city)
            parts.append(self.say(f"Delivery {facts['eta_days'][0]} se {facts['eta_days'][1]} working days mein hoti hai.",
                                  f"Delivery takes {facts['eta_days'][0]} to {facts['eta_days'][1]} working days."))
        if "charge" in topics:
            if s.city:
                facts["delivery_charge"] = delivery_charge(s.city, 0)
                parts.append(self.say(f"{s.city} ka delivery charge {rs(facts['delivery_charge'])} hai (Rs. 10,000 ya zyada ke order par free).",
                                      f"Delivery to {s.city} costs {rs(facts['delivery_charge'])} (free on orders of Rs. 10,000 or more)."))
            else:
                parts.append(self.say("Delivery charge major cities mein Rs. 250 aur baqi shehron mein Rs. 350 hai; Rs. 10,000 ya zyada par free.",
                                      "Delivery is Rs. 250 for major cities and Rs. 350 elsewhere; free on Rs. 10,000 or more."))
        if "cod" in topics:
            facts["cod"] = rules["payment"]["cod_available"]
            parts.append(self.say("COD har delivery area mein available hai (Rs. 25,000 se upar ke orders ki payment staff confirm karta hai).",
                                  "COD is available everywhere (orders above Rs. 25,000 are confirmed by staff first)."))
        if "hours" in topics:
            facts["human_hours"] = rules["human_hours"]
            parts.append(self.say("Hamara staff roz 10:00 AM se 8:00 PM tak available hota hai.", "Our staff is available daily 10:00 AM - 8:00 PM."))
        if "online" in topics:
            facts["online_only"] = True
            parts.append(self.say("Hum sirf online hain, koi showroom nahi hai.", "We are online only; there is no showroom."))
        if "abroad" in topics:
            facts["delivers_abroad"] = False
            parts.append(self.say("Pakistan se bahar delivery available nahi hai.", "We do not deliver outside Pakistan."))
        if "free" in topics:
            facts["free_delivery_min"] = d["free_delivery_min_product_total"]
            parts.append(self.say("Rs. 10,000 ya zyada ke product total par delivery free hai.", "Delivery is free for product totals of Rs. 10,000 or more."))
        if "exchange" in topics:
            ex = rules["exchange"]
            facts.update(window_days=ex["window_days"], sale_items_allowed=ex["sale_items_allowed"])
            parts.append(self.say("Exchange delivery ke 7 din ke andar ho sakta hai, sirf same product ka size/color, unused aur tags ke saath. Sale items exchange nahi hotay.",
                                  "Exchange is possible within 7 days of delivery for another size/color of the same unused product with tags. Sale items cannot be exchanged."))
        facts["source"] = source
        return self.act("answer_policy", " ".join(parts), facts)

    # ---------- recommendation
    def do_recommend(self, raw):
        s = self.s
        if not s.budget:
            return self.act("request_missing_info", self.say("Aap ka budget kitna hai?", "What is your budget?"), {"missing_fields": ["budget"]})
        lo, hi = round(s.budget * 0.75), round(s.budget * 1.25)
        df = catalog[(catalog["price"] >= lo) & (catalog["price"] <= hi)]
        df = df[[self.store.stock[k] > 0 for k in df["sku"]]]
        if s.cat_hint:
            df = df[df["category"] == s.cat_hint]
        picks = (df.assign(gap=(df["price"] - s.budget).abs()).sort_values(["gap", "name"]).drop_duplicates("name").head(3))
        result = {"price_range": [lo, hi], "in_stock_only": True, "picks": picks["name"].tolist()}
        self.store._log("recommend", (s.budget, s.cat_hint), result)
        if picks.empty:
            return self.act("recommend", self.say("Is budget mein koi item in stock nahi mila. Kya budget thora barha sakte hain?",
                                                  "I found nothing in stock in that range. Can you raise the budget a little?"), result)
        lines = "; ".join(f"{r.name} ({rs(r.price)})" for r in picks.itertuples())
        return self.act("recommend", self.say(f"Aap ke budget ({rs(lo)}-{rs(hi)}) mein ye options in stock hain: {lines}.",
                                              f"In your budget ({rs(lo)}-{rs(hi)}) these are in stock: {lines}."), result)

    # ---------- stock check
    def complete_item(self, it):
        v = self.store.variants(it["product"], it["color"])
        if not it["size"] and v["size"].nunique() == 1:
            it["size"] = v["size"].iloc[0]
        v = self.store.variants(it["product"])
        if not it["color"] and v["color"].nunique() == 1:
            it["color"] = v["color"].iloc[0]
        if not it["size"] and it["color"] and self.store.variants(it["product"], it["color"])["size"].nunique() == 1:
            it["size"] = self.store.variants(it["product"], it["color"])["size"].iloc[0]

    def stock_reply(self, it, out):
        name = f"{it['product']} {it['color']} {it['size']}"
        if not out["in_stock"]:
            alt = ", ".join(f"{c} {sz} ({n} pcs)" for c, sz, n in out.get("alternatives", [])) or "koi nahi"
            return self.say(f"Maazrat, {name} abhi out of stock hai. In stock options: {alt}.",
                            f"Sorry, {name} is out of stock. In stock: {alt}.")
        if out.get("requested_exceeds_stock"):
            return self.say(f"{name} ke sirf {out['available']} pieces available hain (price {rs(out['price'])}).",
                            f"Only {out['available']} pieces of {name} are available (price {rs(out['price'])}).")
        return self.say(f"Ji, {name} available hai: {out['available']} pieces, price {rs(out['price'])}.",
                        f"Yes, {name} is available: {out['available']} pieces, price {rs(out['price'])}.")

    def do_stock(self, raw):
        s = self.s
        it = s.items[-1] if s.items else None
        if not it:
            return self.act("request_missing_info", self.say("Kaun sa product chahiye? Naam bata dein.", "Which product do you mean?"),
                            {"missing_fields": ["product"]})
        self.complete_item(it)
        if not it["color"] or not it["size"]:
            v = self.store.variants(it["product"], it["color"], it["size"])
            v = v[[self.store.stock[k] > 0 for k in v["sku"]]]
            opts = ", ".join(sorted({f"{r.color} {r.size}" for r in v.head(8).itertuples()}))
            return self.act("check_stock", self.say(f"{it['product']} ke in-stock options: {opts}. Kaun sa color/size chahiye?",
                                                    f"In-stock options for {it['product']}: {opts}. Which color/size?"),
                            {"in_stock": not v.empty, "available": int(sum(self.store.stock[k] for k in v["sku"])),
                             "missing_fields": [f for f in ("color", "size") if not it[f]]})
        out = self.store.check_stock(it["product"], it["color"], it["size"], it["qty"])
        return self.act("check_stock", self.stock_reply(it, out), out)

    # ---------- order placement
    def do_order(self, raw):
        s = self.s
        confirmed = s.summary_shown and bool(CONFIRM_RE.search(norm(raw))) and not DENY_RE.search(norm(raw))
        if not s.items:
            return self.act("request_missing_info", self.say("Kaun sa product order karna hai?", "Which product would you like to order?"),
                            {"missing_fields": ["product"]})
        for it in s.items:
            self.complete_item(it)
        for it in s.items:
            miss = [f for f in ("product", "color", "size", "qty") if not it[f]]
            if miss:
                return self.act("request_missing_info", self.say(f"Order ke liye ye details chahiye: {', '.join(miss)}.",
                                                                 f"To place the order I still need: {', '.join(miss)}."),
                                {"missing_fields": miss})
        lines, total = [], 0
        for it in s.items:
            out = self.store.check_stock(it["product"], it["color"], it["size"], it["qty"])
            if not out["in_stock"] or out.get("requested_exceeds_stock"):
                return self.act("check_stock", self.stock_reply(it, out), out)
            lines.append((out["sku"], it["qty"], out["price"], it))
            total += out["price"] * it["qty"]
        miss = [k for k, v in (("customer_name", s.name), ("phone", s.phone), ("address", s.address), ("city", s.city)) if not v]
        if miss:
            return self.act("request_missing_info", self.say(f"Order ke liye ye details bhi chahiye: {', '.join(miss)}.",
                                                             f"I also need: {', '.join(miss)}."), {"missing_fields": miss})
        charge = delivery_charge(s.city, total)
        grand = total + charge
        if self.guardrails and grand > rules["payment"]["cod_max_total_without_human"]:
            return self.escalate("cod_above_limit", {"grand_total": grand})
        summary = {"order_created": False, "must_show_summary": True, "product_total": total, "delivery_charge": charge,
                   "grand_total": grand, "payment": "COD"}
        if not confirmed:
            s.summary_shown = True
            items_txt = ", ".join(f"{l[3]['qty']} x {l[3]['product']} {l[3]['color']} {l[3]['size']}" for l in lines)
            return self.act("request_confirmation", self.say(
                f"Order summary: {items_txt}. Product total {rs(total)}, delivery {rs(charge)}, grand total {rs(grand)} (COD). "
                f"Confirm karne ke liye 'haan' likhein.",
                f"Order summary: {items_txt}. Product total {rs(total)}, delivery {rs(charge)}, grand total {rs(grand)} (COD). "
                f"Reply 'yes' to confirm."), summary)
        if not self.store.reserve_order([(l[0], l[1]) for l in lines]):
            return self.act("check_stock", self.say("Maazrat, stock abhi khatam ho gaya.", "Sorry, the stock just ran out."), {"in_stock": False})
        out = self.store.create_order([(l[0], l[1]) for l in lines], s.name, s.phone, s.city, s.address, total, charge)
        out["order_created"] = True
        return self.act("create_order", self.say(
            f"Shukriya! Order #{out['order_id']} confirm ho gaya. Total {rs(grand)} (COD), delivery 2-6 working days mein.",
            f"Thank you! Order #{out['order_id']} is confirmed. Total {rs(grand)} (COD)."), out)

    # ---------- order status
    def need_order_and_phone(self):
        s = self.s
        miss = [k for k, v in (("order_id", s.order_id), ("phone", s.phone)) if not v]
        if miss:
            return self.act("request_missing_info", self.say("Order number aur order par diya gaya phone number bata dein.",
                                                             "Please share your order number and the phone number used on the order."),
                            {"missing_fields": miss})
        return None

    def do_status(self, raw):
        s = self.s
        ask = self.need_order_and_phone()
        if ask:
            return ask
        out = self.store.get_order_status(s.order_id, s.phone)
        if not out["found"]:
            return self.escalate("order_not_found")
        if not out["share_details"] and self.guardrails:
            return self.act("deny_status_mismatch", self.say("Order number aur phone number match nahi hue, is liye details share nahi kar sakte.",
                                                             "The order number and phone number do not match, so I cannot share details."),
                            {"share_details": False})
        if not out["share_details"]:                              # ablation without guardrails: leak the status
            out = self.store.get_order_status(s.order_id, self.store.get_order(s.order_id)["phone"])
        txt = {"Placed": "order mil gaya hai", "Packed": "order pack ho gaya hai", "Shipped": "order courier ko de diya gaya hai",
               "Delivered": "order deliver ho chuka hai"}[out["status"]]
        extra = f" Courier {out['courier']}, tracking {out['tracking_no']}." if "tracking_no" in out else ""
        return self.act("get_order_status", self.say(f"Aap ka {txt}.{extra}", f"Your order status: {out['status']}.{extra}"), out)

    # ---------- exchange
    def do_exchange(self, raw):
        s = self.s
        ask = self.need_order_and_phone()
        if ask:
            return ask
        o = self.store.get_order(s.order_id)
        if o is None:
            return self.escalate("order_not_found")
        if o["phone"] != s.phone and self.guardrails:
            return self.act("deny_status_mismatch", self.say("Order aur phone number match nahi hue.", "Order and phone number do not match."),
                            {"share_details": False})
        _it = SKU.loc[o["sku"]]
        s.exchange_item = {"product": _it["name"], "color": s.target_color or _it["color"], "size": s.target_size or _it["size"], "qty": 1}
        if not (s.target_size or s.target_color or s.different_product):
            return self.act("request_missing_info", self.say("Kaun sa size ya color chahiye?", "Which size or color do you want instead?"),
                            {"missing_fields": ["requested_size_or_color"]})
        if self.guardrails:
            ok, reason, det = exchange_check(self.store, o, s.target_size, s.target_color, s.different_product)
            if not ok:
                return self.escalate(reason, det)
        else:
            item = SKU.loc[o["sku"]]
            rows = self.store.variants(item["name"], s.target_color or item["color"], s.target_size or item["size"])
            det = {"eligible": True, "days_since_delivery": (pd.Timestamp(TODAY) - o["delivered_date"]).days,
                   "replacement_sku": rows.iloc[0]["sku"] if len(rows) else None, "exchange_courier_charge": 250,
                   "product": item["name"], "color": s.target_color or item["color"], "size": s.target_size or item["size"]}
        s.exchange_item = {"product": det["product"], "color": det["color"], "size": det["size"], "qty": 1}
        result = {k: det[k] for k in ("eligible", "days_since_delivery", "replacement_sku", "exchange_courier_charge")}
        self.store._log("create_exchange", (s.order_id, det["replacement_sku"]), result)
        return self.act("create_exchange", self.say(
            f"Exchange request ban gayi hai ({det['product']} {det['color']} {det['size']}). Courier charges {rs(250)} aap ko dene honge; "
            f"item unused aur tags ke saath hona chahiye. Replacement humein purana item milne ke baad bheja jayega.",
            f"Your exchange request is created ({det['product']} {det['color']} {det['size']}). Courier charge {rs(250)}; the item must be unused with tags. "
            f"The replacement ships after we receive the original."), result)


def run_conversation(conv, **cfg):
    ag, turns = Agent(**cfg), []
    for t in conv["turns"]:
        if t["speaker"] == "customer":
            turns.append((t["text"], ag.handle(t["text"])))
    return ag, turns


def show(conv, **cfg):
    ag, turns = run_conversation(conv, **cfg)
    print(f"--- {conv['conv_id']} [{conv['category']}] expected: {conv['expected']['expected_action']}"
          f"{' / ' + conv['expected']['escalation_reason'] if conv['expected']['escalation_reason'] else ''}")
    for text, out in turns:
        print("  Customer:", text.replace("\n", " | "))
        print(f"  Agent   : [{out['action']}] {out['reply']}")
    return ag, turns

### 7. Evaluation harness (metrics per conversation, then aggregated)

In [7]:
def values_equal(a, b):
    if isinstance(a, list):
        return list(b or []) == a
    return a == b


def result_match(gold, pred):
    """Every key of the labelled expected_result must be present and equal in the agent result."""
    return all(k in pred and values_equal(v, pred[k]) for k, v in gold.items())


def item_tuples(items):
    return sorted((i.get("product"), i.get("color"), i.get("size"), i.get("qty")) for i in items)


def evaluate(convs, **cfg):
    rows = []
    for c in convs:
        ag, turns = run_conversation(c, **cfg)
        last, g = turns[-1][1], c["expected"]
        ent, gent = ag.s.entities(), g["entities"]
        pred_esc = last["action"] == "escalate"
        a_ok = last["action"] == g["expected_action"]
        r_ok = (last["reason"] == g["escalation_reason"]) if g["should_escalate"] else True
        res_ok = result_match(g["expected_result"], last["result"]) if a_ok else False
        gi, pi = gent["items"], ent["items"]
        rows.append({
            "conv_id": c["conv_id"], "category": c["category"], "split": c["split"], "difficulty": c["difficulty"],
            "gold_action": g["expected_action"], "pred_action": last["action"],
            "gold_esc": g["should_escalate"], "pred_esc": pred_esc,
            "gold_reason": g["escalation_reason"], "pred_reason": last["reason"],
            "action_ok": a_ok, "esc_ok": pred_esc == g["should_escalate"], "reason_ok": r_ok, "result_ok": res_ok,
            "e2e_ok": a_ok and r_ok and res_ok,
            "items_gold": len(gi) > 0, "items_ok": item_tuples(gi) == item_tuples(pi) if gi else None,
            "phone_ok": (ent["phone"] == gent["phone"]) if gent["phone"] else None,
            "order_ok": (ent["order_id"] == gent["order_id"]) if gent["order_id"] else None,
            "loc_ok": (ent["location"] == gent["location"]) if gent["location"] else None,
            "turns": len(turns)})
    return pd.DataFrame(rows)


def summarize(df, label=""):
    tp = (df.gold_esc & df.pred_esc).sum()
    prec = tp / max(df.pred_esc.sum(), 1)
    rec = tp / max(df.gold_esc.sum(), 1)
    gold_esc = df[df.gold_esc & df.pred_esc]
    out = {"n": len(df),
           "action_acc": df.action_ok.mean(), "esc_precision": prec, "esc_recall": rec,
           "esc_f1": 2 * prec * rec / max(prec + rec, 1e-9),
           "reason_acc": gold_esc.reason_ok.mean() if len(gold_esc) else np.nan,
           "result_acc": df.result_ok.mean(), "e2e_acc": df.e2e_ok.mean(),
           "item_exact": df.items_ok.dropna().astype(float).mean(), "phone_acc": df.phone_ok.dropna().astype(float).mean(),
           "order_id_acc": df.order_ok.dropna().astype(float).mean(), "location_acc": df.loc_ok.dropna().astype(float).mean()}
    return pd.Series(out, name=label)


dev_df = evaluate(dev)
print(summarize(dev_df, "dev").round(3).to_string())
print("\nFailures on dev:")
print(dev_df[~dev_df.e2e_ok][["conv_id", "gold_action", "pred_action", "gold_reason", "pred_reason", "result_ok"]].to_string())

n                120.000
action_acc         1.000
esc_precision      1.000
esc_recall         1.000
esc_f1             1.000
reason_acc         1.000
result_acc         1.000
e2e_acc            1.000
item_exact         0.985
phone_acc          1.000
order_id_acc       1.000
location_acc       1.000

Failures on dev:
Empty DataFrame
Columns: [conv_id, gold_action, pred_action, gold_reason, pred_reason, result_ok]
Index: []


## Part A - Test sets
Dev (120) and test (52) come from Phase 2. The test split was already looked at once in Phase 3, so the errors we found there
were used to make version 2 of the system. That means the v2 test numbers are no longer blind. To get one honest comparison
we also wrote a **new set of 38 conversations** (below) before running v2 on it. We wrote them by hand, using the real catalog and
orders, and we worked out each expected answer from the data on paper, not with our own code.

### 8. Test sets: Phase 3 stress set (20) + new hand-written set (38)

In [8]:
def mk(cid, cat, turns, action, reason=None, result=None):
    return {"conv_id": cid, "category": cat, "split": "stress", "difficulty": "hard",
            "turns": [{"speaker": "customer", "text": t} for t in turns],
            "expected": {"expected_action": action, "should_escalate": action == "escalate", "escalation_reason": reason,
                         "expected_result": result or {},
                         "entities": {"items": [], "phone": None, "order_id": None, "location": None}}}


stress = [
    mk("S01", "stock_check", ["Do you have the olive linen kurti in L?"], "check_stock", result={"in_stock": True}),
    mk("S02", "stock_check", ["black graphic tshirt mdm hai? rate batao"], "check_stock", result={"in_stock": True, "price": 950}),
    mk("S03", "order", ["3 black peshawari chappal size 9 chahiye order karna hai", "Sana Ali 0300-5551212 house 4 street 2 Gulberg Lahore",
                        "haan confirm"], "create_order", result={"order_created": True, "product_total": 10497, "delivery_charge": 0, "grand_total": 10497}),
    mk("S04", "order", ["I want to order 2 black polo t-shirt size L", "Ahmed Khan, 03121234567, house 7 street 1, F-8, Islamabad", "yes please confirm"],
       "create_order", result={"order_created": True, "product_total": 2998, "delivery_charge": 250, "grand_total": 3248}),
    mk("S05", "order", ["white linen kurta M order karna hai"], "request_missing_info"),
    mk("S06", "refund_complaint", ["mujhe refund chahiye, kapra kharab hai"], "escalate", "damaged_item_refund"),
    mk("S07", "handoff_request", ["bhai thora discount kar do"], "escalate", "discount_request"),
    mk("S08", "handoff_request", ["I want to talk to a human please"], "escalate", "customer_requested_human"),
    mk("S09", "refund_complaint", ["mera parcel 3 din se nahi aaya, bohat bura experience hai"], "escalate", "angry_customer"),
    mk("S10", "order", ["polo tshirt black L 11 chahiye"], "escalate", "quantity_above_limit"),
    mk("S11", "handoff_request", ["order cancel karna hai order 10265"], "escalate", "order_cancellation_request"),
    mk("S12", "handoff_request", ["Can I pay via JazzCash?"], "escalate", "online_payment_request"),
    mk("S13", "policy_question", ["kya aap lahore mein cod karte ho?"], "answer_policy", result={"cod": True}),
    mk("S14", "policy_question", ["islamabad delivery kitne din mein hogi?"], "answer_policy", result={"eta_days": [2, 4]}),
    mk("S15", "recommendation", ["budget 1500 ka koi jootay suggest karo"], "recommend", result={"price_range": [1125, 1875], "in_stock_only": True}),
    mk("S16", "policy_question", ["yeh kurta cotton ka hai ya polyester? kya fabric pakka hai?"], "escalate", "outside_policy"),
    mk("S17", "order_status", ["order 10231 status", "03000000000"], "deny_status_mismatch", result={"share_details": False}),
    mk("S18", "order_status", ["where is my order 10231? my number is 0300 1234567"], "get_order_status", result={"status": "Shipped"}),
    mk("S19", "exchange", ["order 10198 exchange karna hai, 03214567890, XL chahiye"], "create_exchange", result={"eligible": True}),
    mk("S20", "exchange", ["bhai order 10120 ka size chota hai, 03335551234, L mein badal do"], "escalate", "sale_item_not_exchangeable"),
]

fresh = [
    # stock checks
    mk("F01", "stock_check", ["Is the navy zip hoodie available in L?"], "check_stock", result={"in_stock": True, "available": 18, "price": 3499}),
    mk("F02", "stock_check", ["chino trouser khaki 32 hai? price kya hai"], "check_stock", result={"in_stock": True, "available": 11, "price": 2699}),
    mk("F03", "stock_check", ["kala zip hoodie M mein hai kya"], "check_stock", result={"in_stock": False, "available": 0, "suggest_alternatives": True}),
    mk("F04", "stock_check", ["blk polo tshrt L hai?"], "check_stock", result={"in_stock": True, "available": 6, "price": 1499}),
    mk("F05", "stock_check", ["pullover sweater grey M ka rate batao"], "check_stock", result={"in_stock": True, "available": 12, "price": 2400}),
    mk("F06", "stock_check", ["leather belt black 34 available?"], "check_stock", result={"in_stock": True, "available": 23, "price": 1299}),
    mk("F07", "stock_check", ["plain cotton kurti pink M hai?"], "check_stock", result={"in_stock": False, "suggest_alternatives": True}),
    # orders
    mk("F08", "order", ["2 white cotton kurta L order karna hai", "Hina Tariq, 0321-9876543, house 5 street 3 Gulberg Lahore", "haan confirm"],
       "create_order", result={"order_created": True, "product_total": 3998, "delivery_charge": 250, "grand_total": 4248, "payment": "COD"}),
    mk("F09", "order", ["4 navy khussa embroidered size 38 chahiye, order karna hai", "Bilal Hussain 03331112223 house 9 street 2 Saddar Karachi", "ji confirm"],
       "create_order", result={"order_created": True, "product_total": 9996, "delivery_charge": 350, "grand_total": 10346, "payment": "COD"}),
    mk("F10", "order", ["4 navy zip hoodie L chahiye bhej do", "Sana Iqbal, 03451230001, house 3 street 1 Cantt Multan", "yes confirm"],
       "create_order", result={"order_created": True, "product_total": 13996, "delivery_charge": 0, "grand_total": 13996, "payment": "COD"}),
    mk("F11", "order", ["6 abaya classic black 54 order karna hai", "Maria Khan 03007778889 house 1 street 2 DHA Lahore", "haan confirm"],
       "escalate", "cod_above_limit", result={"grand_total": 29994}),
    mk("F12", "order", ["lilac lawn 2-piece stitched M order kar do", "Rida Fatima 03114445556 house 7 street 6 Satellite Town Quetta"],
       "request_missing_info", result={"missing_fields": ["qty"]}),   # label corrected after the first run: no quantity was given, same as F13
    mk("F13", "order", ["chino trouser khaki 32 order karna hai"], "request_missing_info", result={"missing_fields": ["qty"]}),
    mk("F14", "order", ["jersey hijab black 15 pieces chahiye"], "escalate", "quantity_above_limit"),
    mk("F15", "order", ["casual shirt beige XL 5 order karna hai"], "check_stock", result={"requested_exceeds_stock": True, "available": 3}),
    mk("F16", "order", ["I want to buy 2 kids frock lawn pink 4-5Y", "Name: Zoya Ali, phone: 03221234000, address: house 12 street 8 Satellite Town, Rawalpindi",
                        "yes, confirm please"], "create_order",
       result={"order_created": True, "product_total": 3998, "delivery_charge": 250, "grand_total": 4248, "payment": "COD"}),
    # policy
    mk("F17", "policy_question", ["Quetta mein COD hai aur charges kitne?"], "answer_policy", result={"cod": True, "delivery_charge": 350}),
    mk("F18", "policy_question", ["free delivery kab hoti hai?"], "answer_policy", result={"free_delivery_min": 10000}),
    mk("F19", "policy_question", ["sale item exchange ho jata hai kya?"], "answer_policy", result={"sale_items_allowed": False}),
    mk("F20", "policy_question", ["staff kitne baje tak reply karta hai?"], "answer_policy", result={"human_hours": "10:00-20:00 PKT"}),
    # order status
    mk("F21", "order_status", ["order 10265 ka status? number 03007654321"], "get_order_status", result={"status": "Placed"}),
    mk("F22", "order_status", ["where is my order 10324? phone 0303 6295863"], "get_order_status", result={"status": "Shipped"}),
    mk("F23", "order_status", ["order 10316 ka status batao", "03001112222"], "deny_status_mismatch", result={"share_details": False}),
    mk("F24", "order_status", ["order 12345 kahan hai? 03001234567"], "escalate", "order_not_found"),
    # exchange
    mk("F25", "exchange", ["order 10373 exchange karna hai, 03380133252, L size chahiye"], "create_exchange",
       result={"eligible": True, "days_since_delivery": 1, "replacement_sku": "SKU0011", "exchange_courier_charge": 250}),
    mk("F26", "exchange", ["order 10355 ka size badalna hai", "03377044104", "L chahiye"], "create_exchange",
       result={"eligible": True, "replacement_sku": "SKU0116"}),
    mk("F27", "exchange", ["order 10350 exchange karna hai, 03014613380, color change chahiye blue"], "escalate", "category_not_exchangeable"),
    mk("F28", "exchange", ["order 10359 exchange karna hai, 03125719294, size 34 chahiye"], "escalate", "outside_7_day_window"),
    # handoff
    mk("F29", "handoff_request", ["mujhe wakeel se baat karni hai, aap ki service kharab hai"], "escalate", "angry_customer_legal_threat"),
    mk("F30", "handoff_request", ["agar main 3 suit lun to kuch discount milega?"], "escalate", "discount_request"),
    mk("F31", "refund_complaint", ["mujhe order wapas karna hai"], "escalate", "refund_request"),
    mk("F32", "handoff_request", ["owner se baat karwa dein"], "escalate", "customer_requested_human"),
    # recommend
    mk("F33", "recommendation", ["2000 tak ka koi sweater dikhao"], "recommend", result={"price_range": [1500, 2500], "in_stock_only": True}),
    mk("F34", "recommendation", ["budget 3500 hai, shirt ya t-shirt batao"], "recommend", result={"price_range": [2625, 4375], "in_stock_only": True}),
    # outside policy / unclear
    mk("F35", "policy_question", ["kya yeh kurta dry clean ho sakta hai?"], "escalate", "outside_policy"),
    mk("F36", "policy_question", ["size chart bhej dein please"], "escalate", "outside_policy"),
    mk("F37", "policy_question", ["kya aap ke paas school uniform hain?"], "escalate", "outside_policy"),
    mk("F38", "handoff_request", ["hi", "kuch dikhao", "acha"], "escalate", "unclear_after_two_attempts"),
]
print(f"stress set: {len(stress)} conversations, new hand-written set: {len(fresh)} conversations")

stress set: 20 conversations, new hand-written set: 38 conversations


### 9. Phase 3 (v1) vs Phase 4 (v2) on every set

In [9]:
def run_all(version, **cfg):
    set_version(version)
    out = {"dev": evaluate(dev, **cfg), "test": evaluate(test, **cfg), "stress": evaluate(stress, **cfg), "new": evaluate(fresh, **cfg)}
    return out


R = {1: run_all(1), 2: run_all(2), 3: run_all(3)}
set_version(3)
rows = []
for v in (1, 2, 3):
    for sp in ("dev", "test", "stress", "new"):
        s = summarize(R[v][sp])
        rows.append({"version": f"v{v}", "set": sp, "n": int(s["n"]), "action_acc": s.action_acc, "esc_F1": s.esc_f1,
                     "reason_acc": s.reason_acc, "result_acc": s.result_acc, "end_to_end": s.e2e_acc})
cmp_table = pd.DataFrame(rows)
print(cmp_table.round(3).to_string(index=False))
print("\nv2 was made from the Phase 3 test + stress errors, v3 from the errors on the new set, so each version is only blind on the set it had not seen.")

version    set   n  action_acc  esc_F1  reason_acc  result_acc  end_to_end
     v1    dev 120       1.000   1.000       1.000       1.000       1.000
     v1   test  52       1.000   1.000       0.889       0.981       0.942
     v1 stress  20       0.900   0.875       1.000       0.900       0.900
     v1    new  38       0.763   0.720       1.000       0.763       0.763
     v2    dev 120       1.000   1.000       1.000       1.000       1.000
     v2   test  52       1.000   1.000       1.000       1.000       1.000
     v2 stress  20       1.000   1.000       1.000       1.000       1.000
     v2    new  38       0.763   0.720       1.000       0.763       0.763
     v3    dev 120       1.000   1.000       1.000       1.000       1.000
     v3   test  52       1.000   1.000       1.000       1.000       1.000
     v3 stress  20       1.000   1.000       1.000       1.000       1.000
     v3    new  38       0.947   0.917       1.000       0.947       0.947

v2 was made from the Pha

## Part B - RAG evaluation (retrieval, embedding size, hallucination)
We have no LLM key in our environment, so we cannot run a real "generation without retrieval" experiment. What we can measure
honestly is (1) how often retrieval hands the generator the sentence that contains the answer, compared with giving it nothing,
a random chunk, or the whole policy; (2) how the embedding size and chunk size change retrieval; and (3) how often the system
answers a question that the policy does not cover (that is the hallucination risk), with and without the retrieval-score gate.

### 10. Policy question sets (every gold fact is checked to really exist in policy.md)

In [10]:
SECTION_OF_KEY = {"eta_days": 2, "delivery_charge": 2, "free_delivery_min": 2, "delivers_abroad": 2, "cod": 3,
                  "window_days": 6, "sale_items_allowed": 6, "human_hours": 1, "online_only": 1}

# (question, gold section, a sentence fragment from policy.md that answers it)
POLICY_QA = [
    ("Lahore mein delivery ka kitna charge hai?", 2, "Rs. 250"),
    ("Quetta ka delivery charge kitna hoga?", 2, "Rs. 350"),
    ("kitne ke order par delivery free hai?", 2, "Rs. 10,000"),
    ("Karachi parcel kitne din mein milega?", 2, "4 to 6 working days"),
    ("Islamabad delivery time kitna hai?", 2, "2 to 4 working days"),
    ("kya aap Pakistan se bahar deliver karte hain?", 2, "outside Pakistan is not available"),
    ("delivery charges har item par alag lagte hain?", 2, "once per order"),
    ("Sheikhupura ka charge kitna hai?", 2, "Rs. 250"),
    ("peshawar delivery kitne din mein hoti hai?", 2, "4 to 6 working days"),
    ("Is COD available in my city?", 3, "Cash on Delivery (COD) is available in all delivery areas"),
    ("25000 se zyada ke order par COD ka kya rule hai?", 3, "Rs. 25,000"),
    ("can I pay with JazzCash or Easypaisa?", 3, "JazzCash"),
    ("ek order mein zyada se zyada kitne pieces mil sakte hain?", 4, "Maximum quantity: 10 pieces"),
    ("order confirm karne ke baad stock kitni der reserve rehta hai?", 4, "15 minutes"),
    ("agar confirm na karun to reservation kab khatam hoti hai?", 4, "15 minutes"),
    ("order place karne ke liye kya details chahiye?", 4, "phone number"),
    ("order ka status kaise check karun?", 5, "order number and the phone number"),
    ("order ke kitne status hote hain?", 5, "Placed, Packed, Shipped, Delivered"),
    ("exchange kitne din ke andar ho sakta hai?", 6, "within 7 days of the delivery date"),
    ("kya sale wali cheez exchange ho sakti hai?", 6, "final sale"),
    ("exchange ka courier charge kitna hai?", 6, "Rs. 250 courier charges"),
    ("abaya ya hijab exchange hota hai?", 6, "hijabs and abayas"),
    ("refund kitne din mein aata hai?", 7, "7 to 10 working days"),
    ("kharab item aaye to kitne ghante mein photos bhejni hain?", 7, "48 hours"),
    ("kya AI refund approve kar sakta hai?", 7, "never approves"),
    ("aap ka staff kab tak available hota hai?", 1, "10:00 AM to 8:00 PM"),
    ("kya aap ka showroom hai jahan aa sakun?", 1, "no walk-in showroom"),
    ("aap kahan based hain?", 1, "Gujranwala"),
    ("discount ya bargaining ke liye kis se baat karun?", 8, "bargaining"),
    ("bot kab insan ko chat hand over karta hai?", 8, "escalate to human staff"),
    ("kya delivery date ka wada kiya jata hai?", 9, "Do not promise delivery dates"),
]
OUT_OF_SCOPE = [
    "kya ye kurta pre-shrunk hai?", "is brand ki warranty kitni hai?", "kya aap gift wrapping karte hain?",
    "kya mere size ke liye alteration ho jayegi?", "kapra imported hai ya local?", "Eid sale kab lag rahi hai?",
    "kya aap Instagram giveaway karte hain?", "size chart bhej dein please", "kya yeh suit dry clean ho sakta hai?",
    "aap ke paas school uniform hain?", "kya aap meri shaadi ke liye custom design bana sakte hain?", "owner ka naam kya hai?",
    "kya aap ki Dubai mein branch hai?", "silk ko iron kaise karte hain?", "kya main apni purani kurti sell kar sakta hun?",
    "aaj ka mausam kaisa hai?",
]
for q, sec, fact in POLICY_QA:
    assert fact in policy_text, fact
policy_convs = [c for c in conversations if c["expected"]["expected_action"] == "answer_policy"]
conv_qa = [(conv_text(c), {SECTION_OF_KEY[k] for k in c["expected"]["expected_result"]}, c["split"]) for c in policy_convs]
print(f"in-scope policy questions: {len(POLICY_QA)} hand-written + {len(conv_qa)} from conversations.json; out-of-scope: {len(OUT_OF_SCOPE)}")

in-scope policy questions: 31 hand-written + 21 from conversations.json; out-of-scope: 16


### 11. Retrieval: chunk size x embedding size x sparse/dense mix

In [11]:
def bullet_chunks(text):
    """One chunk per bullet line (heading text is repeated in every chunk)."""
    out = []
    for section in re.split(r"\n(?=## )", text):
        lines = [l.rstrip() for l in section.strip().split("\n") if l.strip()]
        if not lines or not lines[0].startswith("## "):
            continue
        title = lines[0][3:].strip()
        sec_no = int(re.match(r"(\d+)", title).group(1))
        if sec_no > 9:
            continue
        for l in lines[1:]:
            if len(l.split()) >= 4:
                out.append({"section": sec_no, "title": title, "text": l.strip(" -")})
    return out


class Retriever:
    def __init__(self, chunks, dim=8, alpha=0.5, ngram=(3, 5)):
        self.chunks, self.alpha = chunks, alpha
        docs = [norm(c["title"] + " " + c["text"]) for c in chunks]
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram, sublinear_tf=True)
        self.X = self.vec.fit_transform(docs)
        self.k = max(1, min(dim, self.X.shape[0] - 1, self.X.shape[1] - 1))
        self.svd = TruncatedSVD(n_components=self.k, random_state=SEED)
        self.E = normalize(self.svd.fit_transform(self.X))

    def scores(self, query):
        q = self.vec.transform([norm(query)])
        sparse = (self.X @ q.T).toarray().ravel()
        dense = self.E @ normalize(self.svd.transform(q)).ravel()
        return self.alpha * sparse + (1 - self.alpha) * dense, sparse

    def top(self, query, k=3):
        s, _ = self.scores(query)
        return list(np.argsort(-s)[:k])


ALL_QA = [(q, {sec}, "hand") for q, sec, _ in POLICY_QA] + conv_qa
sections9 = [c for c in chunk_policy(policy_text) if c["section"] <= 9]
chunkers = {"section chunks": sections9, "bullet chunks": bullet_chunks(policy_text)}
print({k: len(v) for k, v in chunkers.items()}, "chunks")


def retrieval_metrics(ret, qa):
    h1 = h3 = mrr = 0
    for q, gold, _ in qa:
        order = ret.top(q, len(ret.chunks))
        secs = [ret.chunks[i]["section"] for i in order]
        h1 += secs[0] in gold
        h3 += bool(gold & set(secs[:3]))
        mrr += 1 / (1 + min(i for i, s in enumerate(secs) if s in gold))
    n = len(qa)
    return {"hit@1": h1 / n, "hit@3": h3 / n, "MRR": mrr / n}


rows = []
for cname, ch in chunkers.items():
    for dim in (2, 4, 8, 16, 32, 64):
        for alpha, mode in ((0.5, "sparse+dense"), (0.0, "dense only"), (1.0, "sparse only")):
            ret = Retriever(ch, dim=dim, alpha=alpha)
            if alpha != 0.5 and dim not in (8, 32):
                continue
            rows.append({"chunks": cname, "dim": ret.k, "mode": mode, **retrieval_metrics(ret, ALL_QA)})
sweep = pd.DataFrame(rows).drop_duplicates(["chunks", "dim", "mode"])
print(f"Retrieval on {len(ALL_QA)} in-scope questions (gold = right policy section):")
print(sweep.round(3).to_string(index=False))

{'section chunks': 9, 'bullet chunks': 54} chunks


Retrieval on 52 in-scope questions (gold = right policy section):
        chunks  dim         mode  hit@1  hit@3   MRR
section chunks    2 sparse+dense  0.731  0.846 0.809
section chunks    4 sparse+dense  0.769  0.942 0.862
section chunks    8 sparse+dense  0.846  0.942 0.905
section chunks    8   dense only  0.846  0.942 0.902
section chunks    8  sparse only  0.846  0.981 0.914
 bullet chunks    2 sparse+dense  0.788  0.885 0.844
 bullet chunks    4 sparse+dense  0.808  0.885 0.854
 bullet chunks    8 sparse+dense  0.788  0.846 0.837
 bullet chunks    8   dense only  0.750  0.827 0.810
 bullet chunks    8  sparse only  0.846  0.962 0.903
 bullet chunks   16 sparse+dense  0.846  0.923 0.890
 bullet chunks   32 sparse+dense  0.846  0.962 0.900
 bullet chunks   32   dense only  0.846  0.962 0.900
 bullet chunks   32  sparse only  0.846  0.962 0.903
 bullet chunks   53 sparse+dense  0.846  0.962 0.903


### 12. Retrieval vs no retrieval: does the generator get the sentence that holds the answer?

In [12]:
def fact_recall(ret, k):
    hits = 0
    for q, sec, fact in POLICY_QA:
        text = " ".join(ret.chunks[i]["text"] for i in ret.top(q, k))
        hits += fact in text
    return hits / len(POLICY_QA)


best = {name: Retriever(ch, dim=32, alpha=0.5) for name, ch in chunkers.items()}
rng = np.random.default_rng(SEED)
ctx_rows = [{"context given to the generator": "nothing (closed book)", "answer sentence present": 0.0, "words of context": 0}]
for name, ret in best.items():
    avg_words = np.mean([len(ret.chunks[i]["text"].split()) for q, _, _ in POLICY_QA for i in ret.top(q, 3)]) * 3
    rand = np.mean([np.mean([POLICY_QA[j][2] in " ".join(ret.chunks[i]["text"] for i in rng.choice(len(ret.chunks), 3, replace=False))
                             for j in range(len(POLICY_QA))]) for _ in range(30)])
    ctx_rows += [{"context given to the generator": f"random 3 {name}", "answer sentence present": rand, "words of context": round(avg_words)},
                 {"context given to the generator": f"retrieved top-1 ({name})", "answer sentence present": fact_recall(ret, 1),
                  "words of context": round(avg_words / 3)},
                 {"context given to the generator": f"retrieved top-3 ({name})", "answer sentence present": fact_recall(ret, 3), "words of context": round(avg_words)}]
ctx_rows.append({"context given to the generator": "whole policy in the prompt", "answer sentence present": 1.0, "words of context": len(policy_text.split())})
ctx_table = pd.DataFrame(ctx_rows)
print(ctx_table.round(3).to_string(index=False))

  context given to the generator  answer sentence present  words of context
           nothing (closed book)                    0.000                 0
         random 3 section chunks                    0.370               362
retrieved top-1 (section chunks)                    0.935               121
retrieved top-3 (section chunks)                    1.000               362
          random 3 bullet chunks                    0.069                47
 retrieved top-1 (bullet chunks)                    0.548                16
 retrieved top-3 (bullet chunks)                    0.839                47
      whole policy in the prompt                    1.000              1313


### 13. Hallucination risk: does the system answer questions the policy does not cover?

In [13]:
from sklearn.metrics import roc_auc_score

ret_cal = Retriever(sections9, dim=8, alpha=0.5)             # the retriever the agent uses
word_vec = TfidfVectorizer(analyzer="word", sublinear_tf=True).fit([norm(c["title"] + " " + c["text"]) for c in sections9])
Xw = word_vec.transform([norm(c["title"] + " " + c["text"]) for c in sections9])
STOP = set("kya hai hain ho ka ki ke ko se mein me aur ya aap main mera kar karna karte karta kitna kitne kab kahan koi yeh ye wo please bhej".split())


def gate_scores(q):
    s_h, s_s = ret_cal.scores(q)
    s_w = (Xw @ word_vec.transform([norm(q)]).T).toarray().ravel()
    top = int(np.argmax(s_h))
    qw = {w for w in tokens(norm(q)) if len(w) >= 4 and w not in STOP}
    cw = set(tokens(norm(sections9[top]["text"])))
    overlap = sum(any(w == x or (len(w) >= 6 and fuzz.ratio(w, x) >= 85) for x in cw) for w in qw)
    return {"char sparse (used in v1/v2)": float(s_s.max()), "hybrid sparse+dense": float(s_h.max()),
            "word TF-IDF": float(s_w.max()), "word overlap with top chunk": float(overlap)}


cal_in = [conv_text(c) for c in dev if c["expected"]["expected_action"] == "answer_policy"]
cal_out = [conv_text(c) for c in dev if c["expected"]["escalation_reason"] == "outside_policy"]
test_in = [q for q, _, _ in POLICY_QA]
test_out = OUT_OF_SCOPE
G = {name: pd.DataFrame([gate_scores(q) for q in qs]) for name, qs in
     {"cal_in": cal_in, "cal_out": cal_out, "test_in": test_in, "test_out": test_out}.items()}
rows = []
for sc in G["cal_in"].columns:
    y_cal = [1] * len(G["cal_in"]) + [0] * len(G["cal_out"])
    s_cal = list(G["cal_in"][sc]) + list(G["cal_out"][sc])
    # threshold = the cut that is best on the calibration questions (dev conversations), then judged on the new hand-written ones
    cands = sorted(set(s_cal))
    thr = max(cands, key=lambda t: np.mean([(s >= t) == bool(y) for s, y in zip(s_cal, y_cal)]))
    y_t = [1] * len(G["test_in"]) + [0] * len(G["test_out"])
    s_t = list(G["test_in"][sc]) + list(G["test_out"][sc])
    rows.append({"gate score": sc, "AUROC (dev calib.)": roc_auc_score(y_cal, s_cal), "AUROC (new questions)": roc_auc_score(y_t, s_t),
                 "threshold": thr, "answers out-of-scope (hallucination)": float(np.mean(np.array(G["test_out"][sc]) >= thr)),
                 "wrongly abstains in-scope": float(np.mean(np.array(G["test_in"][sc]) < thr))})
gate_table = pd.DataFrame(rows)
print(f"calibration questions: {len(cal_in)} in-scope / {len(cal_out)} out-of-scope (dev); new questions: {len(test_in)} in-scope / {len(test_out)} out-of-scope")
print(gate_table.round(3).to_string(index=False))
print("\nNo gate at all (always answer from the best chunk, the usual naive RAG): hallucination on the 16 out-of-scope questions = 100%")

calibration questions: 15 in-scope / 2 out-of-scope (dev); new questions: 31 in-scope / 16 out-of-scope
                 gate score  AUROC (dev calib.)  AUROC (new questions)  threshold  answers out-of-scope (hallucination)  wrongly abstains in-scope
char sparse (used in v1/v2)               1.000                  0.968      0.115                                 0.250                      0.065
        hybrid sparse+dense               1.000                  0.883      0.459                                 0.438                      0.129
                word TF-IDF               0.833                  0.864      0.090                                 0.688                      0.065
word overlap with top chunk               0.883                  0.952      1.000                                 0.375                      0.000

No gate at all (always answer from the best chunk, the usual naive RAG): hallucination on the 16 out-of-scope questions = 100%


### 13b. Same question, but through the whole agent: does the retrieval gate stop invented answers?

In [14]:
rows = []
for v in (1, 3):
    for gate in (True, False):
        set_version(v)
        if not gate:
            RAG_MIN = -1.0                                   # gate off = always answer from the best chunk (naive RAG)
        out_act = [Agent().handle(q)["action"] for q in OUT_OF_SCOPE]
        in_act = [Agent().handle(q)["action"] for q, _, _ in POLICY_QA]
        rows.append({"version": f"v{v}", "retrieval gate": "on" if gate else "off",
                     "out-of-scope: answered from policy (bad)": np.mean([a == "answer_policy" for a in out_act]),
                     "out-of-scope: handed to staff (good)": np.mean([a == "escalate" for a in out_act]),
                     "out-of-scope: asked to clarify": np.mean([a == "clarify" for a in out_act]),
                     "in-scope: answered": np.mean([a == "answer_policy" for a in in_act])})
set_version(3)
agent_gate = pd.DataFrame(rows)
print(agent_gate.round(3).to_string(index=False))

version retrieval gate  out-of-scope: answered from policy (bad)  out-of-scope: handed to staff (good)  out-of-scope: asked to clarify  in-scope: answered
     v1             on                                     0.500                                 0.438                           0.062               0.613
     v1            off                                     0.938                                 0.000                           0.062               0.645
     v3             on                                     0.250                                 0.688                           0.062               0.677
     v3            off                                     0.750                                 0.188                           0.062               0.677


## Part C - Full metrics and comparison of approaches

### 14. Detailed metrics for the final system (v3) on all 172 labelled conversations

In [15]:
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support

set_version(3)
both = pd.concat([R[3]["dev"], R[3]["test"]], ignore_index=True)
labels = sorted(set(both.gold_action) | set(both.pred_action))
p, r, f, sup = precision_recall_fscore_support(both.gold_action, both.pred_action, labels=labels, zero_division=0)
per_action = pd.DataFrame({"precision": p, "recall": r, "F1": f, "support": sup}, index=labels)
print("Final action, dev + test (172 conversations):")
print(per_action.round(3).to_string())
print(f"macro-F1 = {f1_score(both.gold_action, both.pred_action, average='macro'):.3f}")


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


for name, df in [("dev", R[3]["dev"]), ("test", R[3]["test"]), ("stress", R[3]["stress"]), ("new (not blind for v3)", R[3]["new"])]:
    lo, hi = wilson(int(df.e2e_ok.sum()), len(df))
    print(f"end-to-end on {name:24s}: {df.e2e_ok.mean():.3f}  (95% interval {lo:.2f} to {hi:.2f}, n={len(df)})")
lo, hi = wilson(int(R[2]["new"].e2e_ok.sum()), len(R[2]["new"]))
print(f"end-to-end of v2 on the new set (this is the blind estimate): {R[2]['new'].e2e_ok.mean():.3f}  (95% interval {lo:.2f} to {hi:.2f})")

# entity extraction quality per field (only conversations where the number of gold items matches the predicted number)
field_hits, field_n = Counter(), Counter()
for c in dev + test:
    ag, _ = run_conversation(c)
    g, pr = item_tuples(c["expected"]["entities"]["items"]), item_tuples(ag.s.entities()["items"])
    if g and len(g) == len(pr):
        for gi, pi in zip(g, pr):
            for k, name in enumerate(("product", "color", "size", "qty")):
                field_n[name] += 1; field_hits[name] += gi[k] == pi[k]
print("entity accuracy per field:", {k: round(field_hits[k] / field_n[k], 3) for k in field_n}, "| conversations used:", field_n["product"])

Final action, dev + test (172 conversations):
                      precision  recall   F1  support
answer_policy               1.0     1.0  1.0       21
check_stock                 1.0     1.0  1.0       32
create_exchange             1.0     1.0  1.0        9
create_order                1.0     1.0  1.0       24
deny_status_mismatch        1.0     1.0  1.0        5
escalate                    1.0     1.0  1.0       54
get_order_status            1.0     1.0  1.0       10
recommend                   1.0     1.0  1.0        9
request_confirmation        1.0     1.0  1.0        5
request_missing_info        1.0     1.0  1.0        3
macro-F1 = 1.000
end-to-end on dev                     : 1.000  (95% interval 0.97 to 1.00, n=120)
end-to-end on test                    : 1.000  (95% interval 0.93 to 1.00, n=52)
end-to-end on stress                  : 1.000  (95% interval 0.84 to 1.00, n=20)
end-to-end on new (not blind for v3)  : 0.947  (95% interval 0.83 to 0.99, n=38)
end-to-end of v2 o

entity accuracy per field: {'product': 0.981, 'color': 1.0, 'size': 1.0, 'qty': 1.0} | conversations used: 104


### 15. Comparison of approaches (all trained on dev, scored on the 52 test conversations)

In [16]:
dev_text = [conv_text(c) for c in dev]
direct_clf = make_ml().fit([mask(t) for t in dev_text], [c["expected"]["expected_action"] for c in dev])
direct_pred = direct_clf.predict([mask(conv_text(c)) for c in test])
gold_test = [c["expected"]["expected_action"] for c in test]
rows = [{"approach": "always say 'escalate'", "action_acc": np.mean([g == "escalate" for g in gold_test]),
         "macro_F1": f1_score(gold_test, ["escalate"] * len(test), average="macro"), "end_to_end": 0.0, "note": "no tools, no result"},
        {"approach": "direct action classifier (TF-IDF + LogReg on text)", "action_acc": np.mean(np.array(direct_pred) == np.array(gold_test)),
         "macro_F1": f1_score(gold_test, direct_pred, average="macro"), "end_to_end": 0.0, "note": "predicts the label only, cannot fill totals/status"}]
for name, cfg in [("rules-only router + tools", dict(router="rules")), ("ML-only router + tools", dict(router="ml")),
                  ("hybrid router + tools, no guardrails", dict(router="hybrid", guardrails=False)),
                  ("hybrid router + tools + guardrails (ours)", dict(router="hybrid"))]:
    for v in (1, 3):
        if "ours" not in name and v == 1:
            continue
        set_version(v)
        df = evaluate(test, **cfg)
        rows.append({"approach": name + (" [v1]" if v == 1 else " [v3]" if "ours" in name else ""),
                     "action_acc": df.action_ok.mean(), "macro_F1": f1_score(df.gold_action, df.pred_action, average="macro"),
                     "end_to_end": df.e2e_ok.mean(), "note": ""})
set_version(3)
approach_table = pd.DataFrame(rows)
print(approach_table.round(3).to_string(index=False))
print("\nThe test split was used in Phase 3, so v3 = 1.0 there is not a blind score. Rows without a version tag use v3.")

# the same comparison on the new hand-written set, where v2 had not seen the conversations
rows = []
for name, cfg in [("rules-only router", dict(router="rules")), ("ML-only router", dict(router="ml")),
                  ("hybrid, no guardrails", dict(router="hybrid", guardrails=False)), ("hybrid + guardrails", dict(router="hybrid"))]:
    for v in (2, 3):
        set_version(v)
        df = evaluate(fresh, **cfg)
        rows.append({"approach": name, "version": f"v{v}", "action_acc": df.action_ok.mean(), "end_to_end": df.e2e_ok.mean()})
set_version(3)
print("\nNew hand-written set (38):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

                                          approach  action_acc  macro_F1  end_to_end                                               note
                             always say 'escalate'       0.346     0.057       0.000                                no tools, no result
direct action classifier (TF-IDF + LogReg on text)       0.865     0.709       0.000 predicts the label only, cannot fill totals/status
                         rules-only router + tools       1.000     1.000       1.000                                                   
                            ML-only router + tools       0.962     0.963       0.788                                                   
              hybrid router + tools, no guardrails       0.808     0.737       0.808                                                   
    hybrid router + tools + guardrails (ours) [v1]       1.000     1.000       0.942                                                   
    hybrid router + tools + guardrails (ours) [v


New hand-written set (38):
             approach version  action_acc  end_to_end
    rules-only router      v2       0.789       0.789
    rules-only router      v3       0.974       0.974
       ML-only router      v2       0.711       0.605
       ML-only router      v3       0.842       0.711
hybrid, no guardrails      v2       0.737       0.658
hybrid, no guardrails      v3       0.816       0.816
  hybrid + guardrails      v2       0.763       0.763
  hybrid + guardrails      v3       0.947       0.947


## Part D - Where does it fail, and is it fair?

### 16. Error analysis: every failure of every version, with our reading of the cause

In [17]:
CAUSE = {
    ("v1", "C091"): "reason wording: 'insan ko' not in the human-request list",
    ("v1", "C093"): "bug: '7 piece' read as the product word '3-piece' (quantity lost)",
    ("v1", "C128"): "outside-policy question (gift wrapping) sent to the generic handoff",
    ("v1", "S09"): "angry message without our keywords got a policy answer",
    ("v1", "S16"): "fabric question that names a product was treated as a stock check",
    ("v2", "F02"): "bug: bare size '32' read as quantity 32 -> false 'quantity above limit' escalation",
    ("v2", "F06"): "bug: bare size '34' read as quantity 34",
    ("v2", "F11"): "bare size '54' not recognised, so the COD limit was never reached",
    ("v2", "F13"): "bare size '32' read as quantity",
    ("v2", "F19"): "policy question phrased 'ho jata hai kya' routed to the exchange flow",
    ("v2", "F26"): "'badalna' (swap) not recognised, treated as order status",
    ("v2", "F31"): "refund phrased 'order wapas karna hai' NOT escalated (asked to clarify): safety miss",
    ("v2", "F35"): "'dry clean' question answered from the Exchange chunk: retrieval gate let it through",
    ("v2", "F36"): "'size chart bhej dein' has no '?' so it was treated as unclear, not outside-policy",
    ("v3", "F35"): "still answered from a chunk: gate threshold too low",
    ("v3", "F36"): "still treated as unclear (no '?'); the handoff happens only after 3 unclear turns",
}
err_rows = []
for v in (1, 2, 3):
    for sp in ("dev", "test", "stress", "new"):
        df = R[v][sp]
        for _, r in df[~df.e2e_ok].iterrows():
            err_rows.append({"version": f"v{v}", "set": sp, "conv": r.conv_id, "gold": f"{r.gold_action}/{r.gold_reason}",
                             "predicted": f"{r.pred_action}/{r.pred_reason}", "cause": CAUSE.get((f"v{v}", r.conv_id), "")})
err_table = pd.DataFrame(err_rows)
print(err_table.to_string(index=False))
print("\nFailures per version:", err_table.groupby("version").size().to_dict())
unsafe = []
for v in (1, 2, 3):
    d = pd.concat([R[v][k] for k in ("dev", "test", "stress", "new")])
    unsafe.append({"version": f"v{v}", "conversations": len(d), "failures": int((~d.e2e_ok).sum()),
                   "missed escalations (should hand over, did not)": int((d.gold_esc & ~d.pred_esc).sum()),
                   "needless escalations": int((~d.gold_esc & d.pred_esc).sum())})
unsafe_table = pd.DataFrame(unsafe)
print(unsafe_table.to_string(index=False))

version    set conv                              gold                     predicted                                                                                cause
     v1   test C091 escalate/customer_requested_human          escalate/needs_human                             reason wording: 'insan ko' not in the human-request list
     v1   test C093                   check_stock/nan               check_stock/nan                    bug: '7 piece' read as the product word '3-piece' (quantity lost)
     v1   test C128           escalate/outside_policy          escalate/needs_human                  outside-policy question (gift wrapping) sent to the generic handoff
     v1 stress  S09           escalate/angry_customer             answer_policy/nan                               angry message without our keywords got a policy answer
     v1 stress  S16           escalate/outside_policy               check_stock/nan                    fabric question that names a product was treated as 

### 17. Where does v3 do well and badly? (all 230 labelled conversations)

In [18]:
set_version(3)
allr = pd.concat([R[3][k].assign(set=k) for k in ("dev", "test", "stress", "new")], ignore_index=True)
by_cat = allr.groupby("category").agg(n=("e2e_ok", "size"), e2e=("e2e_ok", "mean"), action=("action_ok", "mean")).round(3)
print(by_cat.to_string())
print("\nBy source of the conversation:")
src_map = {c["conv_id"]: c["source"] for c in conversations}
allr["source"] = allr.conv_id.map(lambda i: src_map.get(i, "hand-written (Phase 4)" if i.startswith("F") else "hand-written (stress)"))
print(allr.groupby("source").e2e_ok.agg(["size", "mean"]).round(3).to_string())

                   n    e2e  action
category                           
exchange          30  1.000   1.000
handoff_request   20  1.000   1.000
order             53  1.000   1.000
order_status      24  1.000   1.000
policy_question   31  0.935   0.935
recommendation    12  1.000   1.000
refund_complaint  19  1.000   1.000
stock_check       41  1.000   1.000

By source of the conversation:
                        size   mean
source                             
hand-written (Phase 4)    38  0.947
hand-written (stress)     20  1.000
hand_crafted              20  1.000
template_generated       152  1.000


### 18. Fairness and robustness check

In [19]:
# (a) groups: language mix, city tier, product group, difficulty tag.  (b) counterfactual edits that should not change the decision.
def urdu_share(conv):
    ws = [w for w in tokens(norm(strip_phones(conv_text(conv)))) if w.isalpha()]
    return sum(w in URDU_WORDS for w in ws) / max(len(ws), 1)


lab = {c["conv_id"]: c for c in dev + test}
grp = both.copy()
grp["urdu_share"] = grp.conv_id.map(lambda i: urdu_share(lab[i]))
grp["language mix"] = pd.cut(grp.urdu_share, [-0.01, 0.0001, 0.35, 1.0], labels=["English only", "mixed (<35% Urdu words)", "mostly Roman Urdu"])
grp["city tier"] = grp.conv_id.map(lambda i: ("none" if not lab[i]["expected"]["entities"]["location"] else
                                              "tier A (Rs. 250)" if lab[i]["expected"]["entities"]["location"].lower() in TIER_A else "other (Rs. 350)"))


def prod_group(i):
    items = lab[i]["expected"]["entities"]["items"]
    if not items:
        return "no product"
    cat = catalog.loc[catalog["name"] == items[0]["product"], "category"].iloc[0]
    return "Men" if cat.startswith("Men") else "Women" if cat.startswith("Women") else "Kids" if cat == "Kids" else "Footwear/Accessories"


grp["product group"] = grp.conv_id.map(prod_group)
grp["tags"] = grp.conv_id.map(lambda i: lab[i]["tags"])
rows = []
for col in ("language mix", "city tier", "product group", "difficulty"):
    for g, d in grp.groupby(col):
        lo, hi = wilson(int(d.e2e_ok.sum()), len(d))
        rows.append({"group": f"{col}: {g}", "n": len(d), "end_to_end": d.e2e_ok.mean(), "95% low": lo, "95% high": hi})
for tag in ("spelling_variant", "phone_format", "multi_turn", "reference"):
    d = grp[grp.tags.map(lambda t: tag in t)]
    if len(d):
        lo, hi = wilson(int(d.e2e_ok.sum()), len(d))
        rows.append({"group": f"tag: {tag}", "n": len(d), "end_to_end": d.e2e_ok.mean(), "95% low": lo, "95% high": hi})
group_table = pd.DataFrame(rows)
print(group_table.round(3).to_string(index=False))

# (b) counterfactual edits on dev + test: does the decision change when only an irrelevant detail changes?
NAMES = {"male Muslim": ["Muhammad Usman", "Ahmed Raza"], "female Muslim": ["Ayesha Siddiqui", "Fatima Noor"],
         "Christian": ["Mary Thomas", "Daniel Masih"], "Hindu": ["Priya Sharma", "Rajesh Kumar"]}


def edit_conv(conv, fn):
    c = json.loads(json.dumps(conv))
    for t in c["turns"]:
        t["text"] = fn(t["text"])
    return c


def swap_name(group):
    def fn(text):
        n = find_name(text)
        return text.replace(n, NAMES[group][0]) if n else text
    return fn


def phone_style(style):
    def fn(text):
        def f(m):
            d = "0" + m.group(1) + m.group(2)
            return {"dash": d[:4] + "-" + d[4:], "space": d[:4] + " " + d[4:], "intl": "+92 " + d[1:4] + " " + d[4:]}[style]
        return PHONE_RE.sub(f, text)
    return fn


def typo(text, seed=[0]):
    rs_ = np.random.default_rng(len(text) + seed[0])
    words = text.split(" ")
    idx = [i for i, w in enumerate(words) if len(re.sub(r"\W", "", w)) >= 6 and w.isalpha()]
    if idx:
        i = int(rs_.choice(idx)); w = words[i]; j = int(rs_.integers(1, len(w) - 1))
        words[i] = w[:j] + w[j + 1:]
    return " ".join(words)


EDITS = {"original": lambda t: t,
         **{f"name -> {g}": swap_name(g) for g in NAMES},
         "phone '0300-1234567'": phone_style("dash"), "phone '0300 1234567'": phone_style("space"), "phone '+92 300 1234567'": phone_style("intl"),
         "greeting + emoji + thanks": lambda t: "Assalam o alaikum bhai \U0001F64F " + t + " shukriya",
         "ALL CAPS": lambda t: t.upper(), "all lowercase": lambda t: t.lower(), "one typo per message": typo}
base_pred = {c["conv_id"]: (lambda df: (df.pred_action.iloc[0], df.pred_reason.iloc[0]))(evaluate([c])) for c in dev + test}
rows = []
for name, fn in EDITS.items():
    df = evaluate([edit_conv(c, fn) for c in dev + test])
    same = np.mean([(r.pred_action, r.pred_reason) == base_pred[r.conv_id] or (r.pred_action == base_pred[r.conv_id][0] and pd.isna(r.pred_reason) and pd.isna(base_pred[r.conv_id][1]))
                    for r in df.itertuples()])
    rows.append({"edit": name, "end_to_end": df.e2e_ok.mean(), "same decision as before the edit": same})
robust_table = pd.DataFrame(rows)
print(robust_table.round(3).to_string(index=False))

# who is hurt most when the input gets messy? (lower case, upper case and one typo, pooled; n counts conversation x edit pairs)
parts = [evaluate([edit_conv(c, EDITS[e]) for c in dev + test])[["conv_id", "e2e_ok"]] for e in ("all lowercase", "ALL CAPS", "one typo per message")]
sdf = pd.concat(parts).merge(grp[["conv_id", "language mix", "city tier", "product group", "difficulty"]].drop_duplicates("conv_id"), on="conv_id")
rows = []
for col in ("language mix", "city tier", "product group", "difficulty"):
    for g, d in sdf.groupby(col):
        rows.append({"group": f"{col}: {g}", "pairs": len(d), "end_to_end under messy input": d.e2e_ok.mean()})
messy_table = pd.DataFrame(rows)
print(messy_table.round(3).to_string(index=False))

                                group   n  end_to_end  95% low  95% high
language mix: mixed (<35% Urdu words)  56         1.0    0.936       1.0
      language mix: mostly Roman Urdu 116         1.0    0.968       1.0
                      city tier: none 125         1.0    0.970       1.0
           city tier: other (Rs. 350)  20         1.0    0.839       1.0
          city tier: tier A (Rs. 250)  27         1.0    0.875       1.0
  product group: Footwear/Accessories  17         1.0    0.816       1.0
                  product group: Kids   3         1.0    0.438       1.0
                   product group: Men  32         1.0    0.893       1.0
                 product group: Women  45         1.0    0.921       1.0
            product group: no product  75         1.0    0.951       1.0
                     difficulty: easy  63         1.0    0.943       1.0
                     difficulty: hard 109         1.0    0.966       1.0
                    tag: phone_format  24         1

                     edit  end_to_end  same decision as before the edit
                 original       1.000                             1.000
      name -> male Muslim       1.000                             1.000
    name -> female Muslim       1.000                             1.000
        name -> Christian       1.000                             1.000
            name -> Hindu       1.000                             1.000
     phone '0300-1234567'       1.000                             1.000
     phone '0300 1234567'       1.000                             1.000
  phone '+92 300 1234567'       1.000                             1.000
greeting + emoji + thanks       0.930                             0.930
                 ALL CAPS       0.901                             0.901
            all lowercase       0.901                             0.901
     one typo per message       0.849                             0.901


                                group  pairs  end_to_end under messy input
language mix: mixed (<35% Urdu words)    168                         0.774
      language mix: mostly Roman Urdu    348                         0.937
                      city tier: none    375                         0.965
           city tier: other (Rs. 350)     60                         0.767
          city tier: tier A (Rs. 250)     81                         0.593
  product group: Footwear/Accessories     51                         0.784
                  product group: Kids      9                         0.444
                   product group: Men     96                         0.865
                 product group: Women    135                         0.874
            product group: no product    225                         0.938
                     difficulty: easy    189                         0.889
                     difficulty: hard    327                         0.881


## Part E - The generator side: harness for the "no retrieval" baseline
We could not run it (no LLM key here). The functions below are what we would run with a key, and we test them with a fake model
that only copies the context, just to prove the scoring code works. Those dry-run numbers are NOT results.

### 19. LLM baseline harness (closed-book vs RAG) - dry run with a stand-in model

In [20]:
STRICT_SYSTEM = SYSTEM_PROMPT + """
GROUNDING: answer only from the CONTEXT block. If the CONTEXT does not contain the answer, say exactly:
"Mujhe is ka jawab policy mein nahi mila, staff aap ko reply karega." Never use outside knowledge about other shops."""
ABSTAIN_RX = re.compile(r"policy mein nahi|staff|not sure|cannot|nahi pata", re.I)


def build_prompt(question, contexts):
    ctx = "\n".join(f"- {c}" for c in contexts) if contexts else "(no context)"
    return f"{STRICT_SYSTEM}\n\nCONTEXT:\n{ctx}\n\nCUSTOMER: {question}\nASSISTANT:"


def eval_llm(llm_fn, use_retrieval, k=1):
    """llm_fn(prompt) -> text. Scores: answer contains the gold fragment (in-scope) and abstains (out-of-scope)."""
    ret = best["section chunks"]
    hit = sum(fact in llm_fn(build_prompt(q, [ret.chunks[i]["text"] for i in ret.top(q, k)] if use_retrieval else [])) for q, _, fact in POLICY_QA)
    abst = sum(bool(ABSTAIN_RX.search(llm_fn(build_prompt(q, [ret.chunks[i]["text"] for i in ret.top(q, k)] if use_retrieval else []))))
               for q in OUT_OF_SCOPE)
    return {"answer has the fact": hit / len(POLICY_QA), "abstains on out-of-scope": abst / len(OUT_OF_SCOPE)}


def copy_context_stub(prompt):                     # stand-in "model": returns the context text, or the abstain sentence when empty
    ctx = prompt.split("CONTEXT:\n")[1].split("\n\nCUSTOMER:")[0]
    return "Mujhe is ka jawab policy mein nahi mila, staff aap ko reply karega." if ctx.startswith("(no context)") else ctx


LLM_FN = None            # <- set this to a function that calls your LLM to run the real baseline
print("dry run of the harness with a stand-in model (not a result):")
print("  with retrieval   :", eval_llm(copy_context_stub, True))
print("  without retrieval:", eval_llm(copy_context_stub, False))
if LLM_FN is not None:
    print("REAL  with retrieval   :", eval_llm(LLM_FN, True))
    print("REAL  without retrieval:", eval_llm(LLM_FN, False))
else:
    print("LLM_FN is None, so the real closed-book vs RAG comparison was not run.")

dry run of the harness with a stand-in model (not a result):
  with retrieval   : {'answer has the fact': 0.9354838709677419, 'abstains on out-of-scope': 0.875}
  without retrieval: {'answer has the fact': 0.0, 'abstains on out-of-scope': 1.0}
LLM_FN is None, so the real closed-book vs RAG comparison was not run.


### 20. Figures for the report

In [21]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
os.makedirs("figures", exist_ok=True)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sec = sweep[(sweep["chunks"] == "section chunks") & (sweep["mode"] == "sparse+dense")]
bul = sweep[(sweep["chunks"] == "bullet chunks") & (sweep["mode"] == "sparse+dense")]
ax[0].plot(sec.dim, sec["hit@1"], "o-", label="section chunks (9)"); ax[0].plot(bul.dim, bul["hit@1"], "s-", label="bullet chunks (54)")
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("embedding size (SVD dimensions)"); ax[0].set_ylabel("hit@1 (right section)")
ax[0].set_title("Retrieval vs embedding size"); ax[0].legend(); ax[0].set_ylim(0.5, 1.0)
for sc, style in [("char sparse (used in v1/v2)", "o-"), ("word overlap with top chunk", "s--")]:
    ths = np.unique(np.concatenate([G["test_in"][sc], G["test_out"][sc]]))
    ax[1].plot([np.mean(G["test_out"][sc] >= t) for t in ths], [np.mean(G["test_in"][sc] < t) for t in ths], style, ms=3, label=sc)
ax[1].set_xlabel("answers out-of-scope question (hallucination)"); ax[1].set_ylabel("wrongly refuses in-scope question")
ax[1].set_title("Retrieval-score gate: trade-off"); ax[1].legend(fontsize=8)
rb = robust_table.set_index("edit")["end_to_end"]
rb.sort_values().plot.barh(ax=ax[2], color="#59A14F"); ax[2].set_title("End-to-end accuracy after harmless edits (dev+test)"); ax[2].set_xlim(0.4, 1.02)
plt.tight_layout(); plt.savefig("figures/phase4_analysis.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved figures/phase4_analysis.png")

saved figures/phase4_analysis.png
